In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

RAW_V3 = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v3"
)

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

SOURCES = [
    "foodclass",
    "lettuce",
    "broccoli",
    "cabbage",
    "onion",
    "eggplant",
    "beef",
    "chicken",
    "pork",
    "fish",
    "shrimp",
]

for name in SOURCES:
    (RAW_V3 / name).mkdir(parents=True, exist_ok=True)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset V3:", RAW_V3)

In [ ]:
from pathlib import Path
import zipfile
import shutil

# Đường dẫn project
PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

RAW_DIR = PROJECT_DIR / "datasets" / "raw"

# Tìm file ZIP có tên chứa FoodClass
zip_files = [
    p for p in RAW_DIR.glob("*.zip")
    if "foodclass" in p.name.lower()
]

if not zip_files:
    raise FileNotFoundError(
        f"Không tìm thấy file ZIP FoodClass trong {RAW_DIR}"
    )

if len(zip_files) > 1:
    raise RuntimeError(
        f"Tìm thấy nhiều file ZIP FoodClass: {zip_files}. "
        "Hãy chọn chính xác một file."
    )

archive = zip_files[0]

print("Đã tìm thấy file:", archive)
print(
    "Dung lượng:",
    round(archive.stat().st_size / 1024**3, 2),
    "GB"
)

# Thư mục giải nén
destination = (
    RAW_DIR
    / "state_sources"
    / "extension_v3"
    / "foodclass"
    / "extracted"
)

destination.mkdir(
    parents=True,
    exist_ok=True
)

# Kiểm tra ZIP trước khi giải nén
with zipfile.ZipFile(archive, "r") as z:

    bad_file = z.testzip()

    if bad_file is not None:
        raise ValueError(
            f"File ZIP lỗi tại: {bad_file}"
        )

    root = destination.resolve()

    total_size = 0

    for member in z.infolist():

        target = (
            root / member.filename
        ).resolve()

        if not target.is_relative_to(root):
            raise ValueError(
                "ZIP chứa đường dẫn không an toàn"
            )

        total_size += member.file_size

    print(
        "Dung lượng sau giải nén dự kiến:",
        round(total_size / 1024**3, 2),
        "GB"
    )

    if total_size > 12 * 1024**3:
        raise RuntimeError(
            "Dung lượng giải nén vượt quá 12 GB"
        )

    free_space = shutil.disk_usage(destination).free

    if free_space < total_size * 1.15:
        raise RuntimeError(
            "Không đủ dung lượng ổ đĩa để giải nén"
        )

    print("Đang giải nén dataset...")

    z.extractall(destination)

print("\nGIẢI NÉN THÀNH CÔNG!")
print("Đường dẫn:", destination)

In [ ]:
from collections import Counter
import pandas as pd

extensions = {
    ".jpg", ".jpeg", ".png", ".webp"
}

counts = Counter()

for image_path in destination.rglob("*"):

    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in extensions:
        continue

    relative_folder = (
        image_path.parent.relative_to(destination)
    )

    counts[str(relative_folder)] += 1

df_foodclass = pd.DataFrame(
    [
        {
            "folder": folder,
            "images": count
        }
        for folder, count in counts.items()
    ]
)

if df_foodclass.empty:
    print("Không tìm thấy ảnh. Cần kiểm tra cấu trúc ZIP.")
else:
    display(
        df_foodclass.sort_values(
            "folder"
        ).reset_index(drop=True)
    )

    print(
        "\nTỔNG SỐ ẢNH:",
        df_foodclass["images"].sum()
    )

    REPORT_DIR = (
        PROJECT_DIR / "datasets" / "reports"
    )

    REPORT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    df_foodclass.to_csv(
        REPORT_DIR / "foodclass_inventory.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Đã lưu báo cáo.")

In [ ]:
from pathlib import Path
from collections import Counter

destination = Path(
    r"C:\Users\ASUS\SIC-Food\datasets\raw"
    r"\state_sources\extension_v3\foodclass\extracted"
)

print("Thư mục tồn tại:", destination.exists())

print("\n===== CÁC FILE VÀ THƯ MỤC ĐẦU TIÊN =====")

for item in list(destination.iterdir())[:30]:
    print(
        "📁" if item.is_dir() else "📄",
        item.name
    )

print("\n===== THỐNG KÊ ĐỊNH DẠNG FILE =====")

extensions_count = Counter()

for path in destination.rglob("*"):
    if path.is_file():
        extensions_count[path.suffix.lower()] += 1

for ext, count in extensions_count.most_common(30):
    print(f"{ext or '[không có đuôi]'}: {count}")

In [ ]:
archive_extensions = {
    ".zip", ".rar", ".7z", ".tar", ".gz"
}

nested_archives = [
    p for p in destination.rglob("*")
    if p.is_file()
    and p.suffix.lower() in archive_extensions
]

print("Số file nén bên trong:", len(nested_archives))

for path in nested_archives[:30]:
    print(
        path.relative_to(destination),
        "-",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

In [ ]:
import zipfile
import shutil

inner_zips = [
    p for p in nested_archives
    if p.suffix.lower() == ".zip"
]

for inner_zip in inner_zips:
    target_dir = inner_zip.parent / f"{inner_zip.stem}_extracted"

    if target_dir.exists() and any(target_dir.iterdir()):
        print("Đã có dữ liệu, bỏ qua:", target_dir)
        continue

    with zipfile.ZipFile(inner_zip) as z:
        if z.testzip() is not None:
            raise ValueError(f"ZIP bị lỗi: {inner_zip}")

        root = target_dir.resolve()
        total_size = 0

        for member in z.infolist():
            target = (root / member.filename).resolve()

            if not target.is_relative_to(root):
                raise ValueError("ZIP có đường dẫn không an toàn")

            total_size += member.file_size

        if total_size > 12 * 1024**3:
            raise ValueError("Dung lượng giải nén vượt giới hạn")

        if shutil.disk_usage(inner_zip.parent).free < total_size * 1.15:
            raise RuntimeError("Ổ đĩa không đủ dung lượng")

        target_dir.mkdir(parents=True, exist_ok=True)
        z.extractall(target_dir)

    print("Đã giải nén:", target_dir)

In [ ]:
from collections import Counter
import pandas as pd

image_extensions = {
    ".jpg", ".jpeg", ".png", ".webp",
    ".bmp", ".tif", ".tiff", ".jfif"
}

counts = Counter()

for path in destination.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        folder = str(path.parent.relative_to(destination))
        counts[folder] += 1

df_foodclass = pd.DataFrame(
    [
        {"folder": folder, "images": count}
        for folder, count in counts.items()
    ],
    columns=["folder", "images"]
)

display(df_foodclass.sort_values("folder"))

print("Tổng ảnh:", df_foodclass["images"].sum())

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

df_full = df_foodclass.sort_values("folder").reset_index(drop=True)

print("DANH SÁCH TOÀN BỘ THƯ MỤC FOODCLASS\n")

for index, row in df_full.iterrows():
    print(f"{index + 1:02d}. {row['folder']}")
    print(f"    Số ảnh: {row['images']:,}")

print("\nTỔNG ẢNH:", df_full["images"].sum())
print("SỐ THƯ MỤC:", len(df_full))

In [ ]:
import re
import unicodedata

TARGETS = {
    "lettuce": ["lettuce", "salad"],
    "broccoli": ["broccoli"],
    "cabbage": ["cabbage"],
    "onion": ["onion"],
    "eggplant": ["eggplant", "aubergine", "brinjal"],
    "beef": ["beef", "cow meat"],
    "chicken": ["chicken", "poultry"],
    "pork": ["pork", "pig meat"],
    "fish": ["fish"],
    "shrimp": ["shrimp", "prawn"],
}

def normalize_name(text):
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    return re.sub(r"[^a-z0-9]+", " ", text).strip()

matches = []

for row in df_foodclass.to_dict("records"):
    folder = row["folder"]
    normalized = normalize_name(folder)

    for ingredient, keywords in TARGETS.items():
        if any(
            re.search(r"\b" + re.escape(k) + r"\b",
                      normalized)
            for k in keywords
        ):
            matches.append({
                "ingredient": ingredient,
                "folder": folder,
                "images": row["images"]
            })

df_matches = pd.DataFrame(
    matches,
    columns=["ingredient", "folder", "images"]
)

display(df_matches)

print("Nguyên liệu tìm thấy:",
      sorted(df_matches["ingredient"].unique()))

print("Nguyên liệu chưa tìm thấy:",
      sorted(set(TARGETS) - set(df_matches["ingredient"])))

In [ ]:
from pathlib import PurePath

STATE_KEYWORDS = {
    "normal_appearance": [
        "fresh", "healthy", "good"
    ],
    "visible_defect": [
        "rotten", "spoiled", "decayed"
    ]
}

audit_rows = []

for row in df_foodclass.to_dict("records"):
    folder = row["folder"]

    # Tránh tìm tên nguyên liệu trong tên dataset cha
    parts = str(folder).replace("\\", "/").split("/")
    class_folder = parts[-1]
    normalized = normalize_name(class_folder)

    ingredient = None

    for target, keywords in TARGETS.items():
        if any(
            re.search(r"\b" + re.escape(k) + r"\b", normalized)
            for k in keywords
        ):
            ingredient = target
            break

    state = "unverified"

    for canonical, keywords in STATE_KEYWORDS.items():
        if any(
            re.search(r"\b" + re.escape(k) + r"\b", normalized)
            for k in keywords
        ):
            state = canonical
            break

    audit_rows.append({
        "folder": folder,
        "class_folder": class_folder,
        "ingredient_candidate": ingredient,
        "state_candidate": state,
        "images": row["images"]
    })

df_audit = pd.DataFrame(audit_rows)

display(
    df_audit[
        df_audit["ingredient_candidate"].notna()
    ].sort_values(
        ["ingredient_candidate", "state_candidate"]
    )
)

print("\nSỐ ẢNH CÓ TÊN NGUYÊN LIỆU TRÙNG DANH SÁCH:")
print(
    df_audit.loc[
        df_audit["ingredient_candidate"].notna(),
        "images"
    ].sum()
)

In [ ]:
REPORT_DIR = (
    PROJECT_DIR / "datasets" / "reports"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

df_audit.to_csv(
    REPORT_DIR / "foodclass_class_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

df_matches.to_csv(
    REPORT_DIR / "foodclass_ingredient_matches.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu báo cáo kiểm kê FoodClass.")

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

# ===== 1. ĐƯỜNG DẪN =====
PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FOODCLASS_ROOT = (
    PROJECT_DIR
    / "datasets"
    / "raw"
    / "state_sources"
    / "extension_v3"
    / "foodclass"
    / "extracted"
)

CSV_PATH = REPORT_DIR / "foodclass_inventory.csv"

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp",
    ".bmp", ".tif", ".tiff", ".jfif"
}

# ===== 2. ĐỌC CSV HOẶC TỰ QUÉT LẠI =====
if CSV_PATH.exists():
    print("Đang đọc báo cáo FoodClass đã lưu...")
    df_foodclass = pd.read_csv(CSV_PATH)

else:
    print("Chưa có CSV. Đang quét lại dataset FoodClass...")

    if not FOODCLASS_ROOT.exists():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục: {FOODCLASS_ROOT}"
        )

    counts = Counter()

    for path in FOODCLASS_ROOT.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            relative = path.parent.relative_to(FOODCLASS_ROOT)
            counts[str(relative)] += 1

    df_foodclass = pd.DataFrame(
        [
            {"folder": folder, "images": count}
            for folder, count in counts.items()
        ],
        columns=["folder", "images"]
    )

    if df_foodclass.empty:
        raise RuntimeError(
            "Không tìm thấy ảnh trong FoodClass"
        )

    df_foodclass.to_csv(
        CSV_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("Đã lưu:", CSV_PATH)

print("\nTổng ảnh:", df_foodclass["images"].sum())
print("Số thư mục nhãn:", len(df_foodclass))


# ===== 3. ÁNH XẠ ĐÚNG CẤU TRÚC FOODCLASS =====

TARGETS = {
    "beef", "chicken", "fish",
    "lettuce", "broccoli", "cabbage",
    "onion", "eggplant", "pork", "shrimp"
}

def parse_foodclass_folder(folder):
    parts = str(folder).replace("\\", "/").split("/")

    if len(parts) < 3:
        return None

    category, source_ingredient, state = parts[-3:]

    category = category.lower().strip()
    source_ingredient = source_ingredient.lower().strip()
    state = state.lower().strip()

    ingredient = None

    if category == "meat":
        ingredient = {
            "beef": "beef",
            "chicken": "chicken",
            "pork": "pork"
        }.get(source_ingredient)

    elif category == "fish":
        if source_ingredient != "fish_eye":
            ingredient = "fish"

    elif category == "vegetables":
        ingredient = {
            "lettuce": "lettuce",
            "broccoli": "broccoli",
            "cabbage": "cabbage",
            "onion": "onion",
            "eggplant": "eggplant"
        }.get(source_ingredient)

    if ingredient not in TARGETS:
        return None

    state_mapping = {
        "fresh": "normal_appearance",
        "rotten": "visible_defect"
    }

    target_state = state_mapping.get(state)

    return {
        "category": category,
        "source_ingredient": source_ingredient,
        "ingredient": ingredient,
        "original_state": state,
        "target_state": target_state,
        "folder": folder
    }


# ===== 4. TẠO BẢNG MAPPING =====

records = []

for row in df_foodclass.to_dict("records"):
    result = parse_foodclass_folder(row["folder"])

    if result is None:
        continue

    result["images"] = int(row["images"])
    records.append(result)

df_foodclass_mapping = pd.DataFrame(
    records,
    columns=[
        "category", "source_ingredient", "ingredient",
        "original_state", "target_state", "folder", "images"
    ]
)

print("\n===== BẢNG ÁNH XẠ FOODCLASS =====")
display(df_foodclass_mapping)

print("\nNguyên liệu tìm thấy:")
print(sorted(df_foodclass_mapping["ingredient"].unique()))

print("\nTổng ảnh ứng viên:")
print(df_foodclass_mapping["images"].sum())

df_foodclass_mapping.to_csv(
    REPORT_DIR / "foodclass_correct_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN THÀNH BƯỚC 52!")

In [ ]:
df_usable = df_foodclass_mapping[
    df_foodclass_mapping["target_state"].notna()
].copy()

df_usable["target_class"] = (
    df_usable["ingredient"]
    + "__"
    + df_usable["target_state"]
)

summary = df_usable.pivot_table(
    index="ingredient",
    columns="target_state",
    values="images",
    aggfunc="sum",
    fill_value=0
)

display(summary)

print(
    "\nTổng ảnh ứng viên cho EfficientNet:",
    df_usable["images"].sum()
)

print("\nCác trạng thái chưa ánh xạ:")
display(
    df_foodclass_mapping[
        df_foodclass_mapping["target_state"].isna()
    ][["source_ingredient", "original_state", "images"]]
)

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display
import random

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

FOODCLASS_ROOT = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v3"
    / "foodclass" / "extracted"
)

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

rng = random.Random(42)
EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".webp", ".bmp", ".tif", ".tiff"
}

for row in df_usable.to_dict("records"):
    folder = FOODCLASS_ROOT / Path(
        row["folder"].replace("\\", "/")
    )

    files = [
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in EXTENSIONS
    ]

    print("\n" + "=" * 50)
    print("Nguyên liệu:", row["ingredient"])
    print("Loại:", row["source_ingredient"])
    print("Trạng thái:", row["original_state"])
    print("Số ảnh:", len(files))

    for path in rng.sample(files, min(2, len(files))):
        with Image.open(path) as im:
            preview = im.convert("RGB")
            preview.thumbnail((280, 280))
            display(preview)

In [ ]:
import hashlib
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

def get_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

records = []
errors = []

for row in df_usable.to_dict("records"):

    folder = FOODCLASS_ROOT / Path(
        row["folder"].replace("\\", "/")
    )

    for path in tqdm(
        sorted(folder.iterdir()),
        desc=row["target_class"]
    ):
        if not path.is_file():
            continue

        if path.suffix.lower() not in EXTENSIONS:
            continue

        try:
            with Image.open(path) as im:
                im.verify()

            records.append({
                "path": str(path),
                "ingredient": row["ingredient"],
                "species": row["source_ingredient"],
                "original_state": row["original_state"],
                "target_class": row["target_class"],
                "sha256": get_sha256(path)
            })

        except Exception as e:
            errors.append({
                "path": str(path),
                "error": str(e)
            })

df_food_raw = pd.DataFrame(records)

print("Tổng ảnh hợp lệ:", len(df_food_raw))
print("Ảnh lỗi:", len(errors))

df_food_raw.to_csv(
    REPORT_DIR / "foodclass_v3_raw_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    errors,
    columns=["path", "error"]
).to_csv(
    REPORT_DIR / "foodclass_v3_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
duplicate_count = df_food_raw.duplicated(
    subset="sha256"
).sum()

label_counts = (
    df_food_raw.groupby("sha256")["target_class"]
    .nunique()
)

conflict_hashes = set(
    label_counts[label_counts > 1].index
)

print("Bản dư trùng SHA256:", duplicate_count)
print("Nhóm trùng nhưng khác nhãn:", len(conflict_hashes))

df_conflicts = df_food_raw[
    df_food_raw["sha256"].isin(conflict_hashes)
]

display(df_conflicts.head(20))

In [ ]:
df_food_candidates = df_food_raw[
    ~df_food_raw["sha256"].isin(conflict_hashes)
].copy()

df_food_candidates = (
    df_food_candidates
    .drop_duplicates(subset="sha256", keep="first")
    .reset_index(drop=True)
)

print("Ảnh ứng viên sau lọc:", len(df_food_candidates))
print("Số lớp:", df_food_candidates["target_class"].nunique())

display(
    df_food_candidates.groupby("target_class")
    .size()
    .reset_index(name="images")
)

df_food_candidates.to_csv(
    REPORT_DIR / "foodclass_v3_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import imagehash
from PIL import Image
from tqdm.auto import tqdm

phashes = []

for path in tqdm(
    df_food_candidates["path"],
    desc="Calculating pHash"
):
    try:
        with Image.open(path) as im:
            phashes.append(
                str(imagehash.phash(im.convert("RGB")))
            )
    except Exception:
        phashes.append(None)

df_food_candidates["phash"] = phashes

print("Ảnh có pHash hợp lệ:",
      df_food_candidates["phash"].notna().sum())

print("Ảnh không tính được pHash:",
      df_food_candidates["phash"].isna().sum())

In [ ]:
df_valid = df_food_candidates.dropna(
    subset=["phash"]
).copy()

hash_class_counts = (
    df_valid.groupby("phash")["target_class"].nunique()
)

ambiguous_hashes = set(
    hash_class_counts[hash_class_counts > 1].index
)

df_ambiguous = df_valid[
    df_valid["phash"].isin(ambiguous_hashes)
]

print("Nhóm pHash giống nhau nhưng khác nhãn:",
      len(ambiguous_hashes))

df_ambiguous.to_csv(
    REPORT_DIR / "foodclass_v3_phash_ambiguities.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_ambiguous.head(20))

In [ ]:
import re
from pathlib import Path
from collections import defaultdict

df_grouped = df_valid[
    ~df_valid["phash"].isin(ambiguous_hashes)
].copy().reset_index(drop=True)

patterns = [
    r"^rotated_by_[+-]?\d+(?:\.\d+)?_",
    r"^vertical_flip_?",
    r"^horizontal_flip_?",
    r"^salt_and_pepper_?",
]

def original_name(filename):
    name = filename.lower()

    changed = True
    while changed:
        changed = False
        for pattern in patterns:
            new = re.sub(pattern, "", name)
            if new != name:
                name = new
                changed = True
    return name

parent = list(range(len(df_grouped)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

seen_phash = {}
seen_name = {}

for i, row in enumerate(df_grouped.to_dict("records")):
    h = row["phash"]

    if h in seen_phash:
        union(i, seen_phash[h])
    else:
        seen_phash[h] = i

    key = (
        row["target_class"],
        row["species"],
        original_name(Path(row["path"]).name)
    )

    if key in seen_name:
        union(i, seen_name[key])
    else:
        seen_name[key] = i

df_grouped["group_id"] = [
    find(i) for i in range(len(df_grouped))
]

print("Số ảnh:", len(df_grouped))
print("Số nhóm ảnh:", df_grouped["group_id"].nunique())

In [ ]:
import random
import pandas as pd

rng = random.Random(42)
split_map = {}

# Chia nhóm theo lớp và loài nguồn
for (class_name, species), part in df_grouped.groupby(
    ["target_class", "species"]
):
    group_ids = sorted(part["group_id"].unique())
    rng.shuffle(group_ids)

    n = len(group_ids)

    if n < 10:
        raise ValueError(
            f"Không đủ nhóm độc lập: {class_name}/{species}: {n}"
        )

    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    for gid in group_ids[:n_train]:
        split_map[gid] = "train"

    for gid in group_ids[n_train:n_train+n_val]:
        split_map[gid] = "val"

    for gid in group_ids[n_train+n_val:]:
        split_map[gid] = "test"

df_grouped["split"] = df_grouped["group_id"].map(split_map)

assert (
    df_grouped.groupby("group_id")["split"].nunique().max()
    == 1
)

summary = pd.crosstab(
    df_grouped["target_class"],
    df_grouped["split"]
)

display(summary)

print("Tổng ảnh:", len(df_grouped))
print("Số lớp:", df_grouped["target_class"].nunique())

df_grouped.to_csv(
    REPORT_DIR / "foodclass_v3_split_plan.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

# Dữ liệu đã chia ở bước 57
df_check = df_grouped.copy().reset_index(drop=True)

def hash_to_bits(value):
    number = int(value, 16)
    return [(number >> i) & 1 for i in range(64)]

bits = np.array(
    [hash_to_bits(h) for h in df_check["phash"]],
    dtype=np.float32
)

splits = df_check["split"].to_numpy()

pairs = []

def compare_splits(split_a, split_b, threshold=5):
    idx_a = np.where(splits == split_a)[0]
    idx_b = np.where(splits == split_b)[0]

    tree = cKDTree(bits[idx_b])

    for i in tqdm(idx_a, desc=f"{split_a} vs {split_b}"):
        neighbors = tree.query_ball_point(
            bits[i],
            r=np.sqrt(threshold) + 1e-6
        )

        for j_local in neighbors:
            j = idx_b[j_local]

            distance = int(
                np.count_nonzero(bits[i] != bits[j])
            )

            if distance <= threshold:
                pairs.append({
                    "image_a": df_check.iloc[i]["path"],
                    "split_a": split_a,
                    "class_a": df_check.iloc[i]["target_class"],
                    "image_b": df_check.iloc[j]["path"],
                    "split_b": split_b,
                    "class_b": df_check.iloc[j]["target_class"],
                    "distance": distance
                })

compare_splits("train", "val")
compare_splits("train", "test")
compare_splits("val", "test")

df_near_food = pd.DataFrame(
    pairs,
    columns=[
        "image_a", "split_a", "class_a",
        "image_b", "split_b", "class_b", "distance"
    ]
)

df_near_food.to_csv(
    REPORT_DIR / "foodclass_v3_near_duplicates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tổng cặp nghi gần trùng:", len(df_near_food))

if not df_near_food.empty:
    display(
        df_near_food.groupby(
            ["split_a", "split_b"]
        ).size().reset_index(name="pairs")
    )

    display(
        df_near_food.groupby("distance")
        .size().reset_index(name="pairs")
    )

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_pairs(df, count=8):
    if df.empty:
        print("Không có cặp nghi trùng.")
        return

    samples = df.sort_values("distance").head(count)

    fig, axes = plt.subplots(
        len(samples), 2,
        figsize=(9, 3.5 * len(samples)),
        squeeze=False
    )

    for index, (_, row) in enumerate(samples.iterrows()):
        for col, key in enumerate(["image_a", "image_b"]):
            with Image.open(row[key]) as img:
                axes[index, col].imshow(img.convert("RGB"))

            split = row["split_a"] if col == 0 else row["split_b"]
            cls = row["class_a"] if col == 0 else row["class_b"]

            axes[index, col].set_title(
                f"{split} | {cls}",
                fontsize=9
            )
            axes[index, col].axis("off")

        axes[index, 0].set_ylabel(
            f"pHash = {row['distance']}"
        )

    plt.tight_layout()
    plt.show()

show_pairs(df_near_food)

In [ ]:
from pathlib import Path

priority = {
    "train": 0,
    "val": 1,
    "test": 2
}

quarantine = set()

for row in df_near_food.to_dict("records"):
    if priority[row["split_a"]] < priority[row["split_b"]]:
        quarantine.add(row["image_b"])
    else:
        quarantine.add(row["image_a"])

df_safe_food = df_grouped[
    ~df_grouped["path"].isin(quarantine)
].copy()

print("Ảnh ban đầu:", len(df_grouped))
print("Ảnh tạm cách ly:", len(quarantine))
print("Ảnh còn lại:", len(df_safe_food))

summary_safe = pd.crosstab(
    df_safe_food["target_class"],
    df_safe_food["split"]
)

display(summary_safe)

required = ["train", "val", "test"]

if not set(required).issubset(summary_safe.columns):
    raise RuntimeError("Thiếu một split.")

if (summary_safe[required] == 0).any().any():
    raise RuntimeError("Có lớp bị thiếu ảnh ở một split.")

# Kiểm tra các cặp đã phát hiện
remaining = set(df_safe_food["path"])

remaining_pairs = df_near_food[
    df_near_food["image_a"].isin(remaining)
    &
    df_near_food["image_b"].isin(remaining)
]

print("Cặp nghi trùng còn lại:", len(remaining_pairs))

assert len(remaining_pairs) == 0

df_safe_food.to_csv(
    REPORT_DIR / "foodclass_v3_safe_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "path": sorted(quarantine)
}).to_csv(
    REPORT_DIR / "foodclass_v3_quarantine.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import shutil
import hashlib
from tqdm.auto import tqdm

OUTPUT_DIR = PROJECT_DIR / "datasets" / "efficientnet_v3_3_clean"

if OUTPUT_DIR.exists() and any(
    p.is_file() for p in OUTPUT_DIR.rglob("*")
):
    raise RuntimeError(
        "Thư mục đầu ra đã có dữ liệu, không ghi đè."
    )

exported = []
errors = []

for row in tqdm(
    df_safe_food.to_dict("records"),
    desc="Export FoodClass V3"
):
    source = Path(row["path"])

    target_dir = (
        OUTPUT_DIR
        / row["split"]
        / row["target_class"]
    )

    target_dir.mkdir(parents=True, exist_ok=True)

    uid = hashlib.sha256(
        str(source.resolve()).encode("utf-8")
    ).hexdigest()[:20]

    target = target_dir / f"{uid}{source.suffix.lower()}"

    try:
        shutil.copy2(source, target)

        exported.append({
            **row,
            "export_path": str(target)
        })

    except Exception as e:
        errors.append({
            "source": str(source),
            "error": str(e)
        })

df_export_food = pd.DataFrame(exported)

df_export_food.to_csv(
    REPORT_DIR / "foodclass_v3_export_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ảnh xuất thành công:", len(exported))
print("Ảnh lỗi:", len(errors))
print("Số lớp:", df_export_food["target_class"].nunique())
print("Đường dẫn:", OUTPUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

DATASET_V2 = PROJECT_DIR / "datasets" / "efficientnet_v2"

FOODCLASS_CLEAN = (
    PROJECT_DIR / "datasets" / "efficientnet_v3_3_clean"
)

DATASET_V3 = PROJECT_DIR / "datasets" / "efficientnet_v3"

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp",
    ".bmp", ".tif", ".tiff", ".jfif"
}

for source in [DATASET_V2, FOODCLASS_CLEAN]:
    print(source, "→", source.exists())

    if not source.exists():
        raise FileNotFoundError(source)

print("\nThư mục V3 sẽ tạo:", DATASET_V3)

In [ ]:
def get_classes(dataset_dir):
    train_dir = dataset_dir / "train"

    return {
        p.name
        for p in train_dir.iterdir()
        if p.is_dir()
    }

classes_v2 = get_classes(DATASET_V2)
classes_new = get_classes(FOODCLASS_CLEAN)

print("Số lớp V2:", len(classes_v2))
print("Số lớp FoodClass:", len(classes_new))

print("\nCác lớp trùng tên:")
print(classes_v2.intersection(classes_new))

all_classes = classes_v2.union(classes_new)

print("\nTổng số lớp:", len(all_classes))

for i, class_name in enumerate(sorted(all_classes)):
    print(f"{i:02d}. {class_name}")

assert len(classes_v2) == 20
assert len(classes_new) == 6
assert classes_v2.isdisjoint(classes_new)
assert len(all_classes) == 26

print("\nKIỂM TRA THÀNH CÔNG!")

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm
import hashlib

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

records = []

for dataset_dir in [DATASET_V2, FOODCLASS_CLEAN]:
    for split in ["train", "val", "test"]:
        split_dir = dataset_dir / split

        for cls in sorted(split_dir.iterdir()):
            if not cls.is_dir():
                continue

            images = [
                p for p in cls.iterdir()
                if p.is_file() and p.suffix.lower() in EXTENSIONS
            ]

            for path in tqdm(
                images,
                desc=f"{dataset_dir.name}/{split}/{cls.name}"
            ):
                records.append({
                    "source_dataset": dataset_dir.name,
                    "source_path": str(path),
                    "split": split,
                    "class": cls.name,
                    "sha256": sha256_file(path)
                })

df_premerge = pd.DataFrame(records)

print("Tổng ảnh được kiểm tra:", len(df_premerge))
print("Số lớp:", df_premerge["class"].nunique())

hash_summary = df_premerge.groupby("sha256").agg(
    splits=("split", "nunique"),
    classes=("class", "nunique")
)

cross_split_hashes = set(
    hash_summary.index[hash_summary["splits"] > 1]
)

conflict_hashes = set(
    hash_summary.index[hash_summary["classes"] > 1]
)

print("Nhóm trùng SHA256 giữa các split:", len(cross_split_hashes))
print("Nhóm trùng SHA256 khác nhãn:", len(conflict_hashes))

df_premerge.to_csv(
    REPORT_DIR / "efficientnet_v3_premerge_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

df_premerge[
    df_premerge["sha256"].isin(
        cross_split_hashes | conflict_hashes
    )
].to_csv(
    REPORT_DIR / "efficientnet_v3_premerge_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm
import hashlib

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

records = []

for dataset_dir in [DATASET_V2, FOODCLASS_CLEAN]:
    for split in ["train", "val", "test"]:
        split_dir = dataset_dir / split

        for cls in sorted(split_dir.iterdir()):
            if not cls.is_dir():
                continue

            images = [
                p for p in cls.iterdir()
                if p.is_file() and p.suffix.lower() in EXTENSIONS
            ]

            for path in tqdm(
                images,
                desc=f"{dataset_dir.name}/{split}/{cls.name}"
            ):
                records.append({
                    "source_dataset": dataset_dir.name,
                    "source_path": str(path),
                    "split": split,
                    "class": cls.name,
                    "sha256": sha256_file(path)
                })

df_premerge = pd.DataFrame(records)

print("Tổng ảnh được kiểm tra:", len(df_premerge))
print("Số lớp:", df_premerge["class"].nunique())

hash_summary = df_premerge.groupby("sha256").agg(
    splits=("split", "nunique"),
    classes=("class", "nunique")
)

cross_split_hashes = set(
    hash_summary.index[hash_summary["splits"] > 1]
)

conflict_hashes = set(
    hash_summary.index[hash_summary["classes"] > 1]
)

print("Nhóm trùng SHA256 giữa các split:", len(cross_split_hashes))
print("Nhóm trùng SHA256 khác nhãn:", len(conflict_hashes))

df_premerge.to_csv(
    REPORT_DIR / "efficientnet_v3_premerge_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

df_premerge[
    df_premerge["sha256"].isin(
        cross_split_hashes | conflict_hashes
    )
].to_csv(
    REPORT_DIR / "efficientnet_v3_premerge_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
if cross_split_hashes or conflict_hashes:
    print("PHÁT HIỆN XUNG ĐỘT DỮ LIỆU!")
    print("Hãy kiểm tra file efficientnet_v3_premerge_conflicts.csv")
    print("Chưa nên gộp và huấn luyện.")
else:
    print("Không phát hiện trùng SHA256 xuyên tập hoặc khác nhãn.")
    print("Có thể chuyển sang Bước 63.")

In [ ]:
import shutil
from tqdm.auto import tqdm

if cross_split_hashes or conflict_hashes:
    raise RuntimeError(
        "Cần xử lý trùng SHA256 hoặc nhãn mâu thuẫn trước khi gộp."
    )

if DATASET_V3.exists() and any(DATASET_V3.rglob("*")):
    raise RuntimeError(
        "Thư mục efficientnet_v3 đã có dữ liệu. "
        "Không ghi đè để tránh mất dữ liệu."
    )

needed_bytes = sum(
    Path(p).stat().st_size
    for p in df_premerge["source_path"]
)

available_bytes = shutil.disk_usage(
    PROJECT_DIR
).free

print("Dung lượng cần (GB):", round(needed_bytes / 1024**3, 2))
print("Dung lượng trống (GB):", round(available_bytes / 1024**3, 2))

if available_bytes < needed_bytes * 1.15:
    raise RuntimeError("Không đủ dung lượng để tạo dataset V3")

export_records = []

for row in tqdm(
    df_premerge.to_dict("records"),
    desc="Merging EfficientNet V3"
):
    source = Path(row["source_path"])

    target_dir = (
        DATASET_V3 / row["split"] / row["class"]
    )

    target_dir.mkdir(parents=True, exist_ok=True)

    target = target_dir / source.name

    if target.exists():
        raise RuntimeError(f"Trùng đường dẫn đích: {target}")

    shutil.copy2(source, target)

    export_records.append({
        **row,
        "target_path": str(target)
    })

df_v3 = pd.DataFrame(export_records)

df_v3.to_csv(
    REPORT_DIR / "efficientnet_v3_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN THÀNH GỘP DATASET!")
print("Tổng ảnh:", len(df_v3))
print("Số lớp:", df_v3["class"].nunique())

display(
    pd.crosstab(df_v3["class"], df_v3["split"])
)

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder(
    DATASET_V3 / "train",
    transform=transform
)

val_ds = datasets.ImageFolder(
    DATASET_V3 / "val",
    transform=transform
)

test_ds = datasets.ImageFolder(
    DATASET_V3 / "test",
    transform=transform
)

assert train_ds.class_to_idx == val_ds.class_to_idx
assert train_ds.class_to_idx == test_ds.class_to_idx

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

print("Tổng ảnh:", len(train_ds) + len(val_ds) + len(test_ds))
print("Số lớp:", len(train_ds.classes))

assert len(train_ds.classes) == 26

loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

images, labels = next(iter(loader))

print("Batch shape:", images.shape)
print("Thiết bị:", "CUDA" if torch.cuda.is_available() else "CPU")

In [ ]:
DATASET_DIR = DATASET_V3

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import imagehash
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_V3 = PROJECT_DIR / "datasets" / "efficientnet_v3"
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp",
    ".bmp", ".tif", ".tiff", ".jfif"
}

records = []
errors = []

for split in ["train", "val", "test"]:
    folder = DATASET_V3 / split

    paths = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    ]

    for path in tqdm(paths, desc=f"pHash {split}"):
        try:
            with Image.open(path) as image:
                h = imagehash.phash(image.convert("RGB"))

            records.append({
                "path": str(path),
                "split": split,
                "class": path.parent.name,
                "phash": str(h)
            })
        except Exception as e:
            errors.append({
                "path": str(path),
                "error": str(e)
            })

df_v3_hash = pd.DataFrame(records)

df_v3_hash.to_csv(
    REPORT_DIR / "efficientnet_v3_phash.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ảnh kiểm tra:", len(df_v3_hash))
print("Ảnh lỗi:", len(errors))

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

def phash_bits(value):
    n = int(value, 16)
    return [(n >> i) & 1 for i in range(64)]

bits = np.array(
    [phash_bits(h) for h in df_v3_hash["phash"]],
    dtype=np.float32
)

split_values = df_v3_hash["split"].to_numpy()
class_values = df_v3_hash["class"].to_numpy()

pairs = []

def compare_split_pair(a, b, threshold=5):
    for class_name in sorted(set(class_values)):
        idx_a = np.where(
            (split_values == a) & (class_values == class_name)
        )[0]

        idx_b = np.where(
            (split_values == b) & (class_values == class_name)
        )[0]

        if not len(idx_a) or not len(idx_b):
            continue

        tree = cKDTree(bits[idx_b])

        for i in idx_a:
            neighbors = tree.query_ball_point(
                bits[i],
                r=np.sqrt(threshold) + 1e-6
            )

            for local_j in neighbors:
                j = idx_b[local_j]

                distance = int(
                    np.count_nonzero(bits[i] != bits[j])
                )

                if distance <= threshold:
                    pairs.append({
                        "image_a": df_v3_hash.iloc[i]["path"],
                        "image_b": df_v3_hash.iloc[j]["path"],
                        "split_a": a,
                        "split_b": b,
                        "class": class_name,
                        "distance": distance
                    })

for a, b in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]:
    compare_split_pair(a, b)

df_v3_near = pd.DataFrame(pairs)

print("Cặp nghi trùng xuyên tập:", len(df_v3_near))

if not df_v3_near.empty:
    display(
        df_v3_near.groupby(["split_a", "split_b"])
        .size()
        .reset_index(name="pairs")
    )

    display(
        df_v3_near.groupby("distance")
        .size()
        .reset_index(name="pairs")
    )

df_v3_near.to_csv(
    REPORT_DIR / "efficientnet_v3_near_duplicates.csv",
    index=False,
    encoding="utf-8-sig"
)

# Kiểm tra riêng pHash giống hệt nhưng bị gán khác lớp
conflicting_hashes = (
    df_v3_hash.groupby("phash")["class"].nunique()
)

conflicting_hashes = set(
    conflicting_hashes[conflicting_hashes > 1].index
)

print("Nhóm pHash giống nhau nhưng khác lớp:", len(conflicting_hashes))

df_v3_hash[
    df_v3_hash["phash"].isin(conflicting_hashes)
].to_csv(
    REPORT_DIR / "efficientnet_v3_phash_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

RAW_V4 = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
)

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

sources = [
    ("lettuce", "healthy-lettuce-vskiy", "detection"),
    ("cabbage", "cabbage-gf0rt", "detection"),
    ("eggplant", "detectvagetable-2yxmr", "detection"),
    ("broccoli", "detectvagetable-2yxmr", "detection"),
    ("onion", "onion-sorting-cghvq", "detection"),
    ("pork", "meat-checker-model-vekso", "detection"),
    ("shrimp", "shrimp-olx4p", "classification"),
]

for ingredient, dataset, task in sources:
    (RAW_V4 / ingredient).mkdir(
        parents=True,
        exist_ok=True
    )

df_sources = pd.DataFrame(
    sources,
    columns=["ingredient", "dataset", "task"]
)

df_sources["status"] = "not_downloaded"
df_sources["labels_verified"] = False

REPORT_DIR.mkdir(parents=True, exist_ok=True)

df_sources.to_csv(
    REPORT_DIR / "efficientnet_v4_sources.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_sources)
print("Đã tạo thư mục:", RAW_V4)

In [ ]:
%pip install roboflow pyyaml

In [ ]:
from pathlib import Path
import zipfile
import shutil

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
RAW_DIR = PROJECT_DIR / "datasets" / "raw"

RAW_V4 = (
    RAW_DIR / "state_sources" / "extension_v4"
)

CABBAGE_DIR = RAW_V4 / "cabbage"
CABBAGE_DIR.mkdir(parents=True, exist_ok=True)

archive = RAW_DIR / "cabbage.v2i.yolov8.zip"

if not archive.exists():
    raise FileNotFoundError(archive)

print("File ZIP:", archive)
print("Dung lượng MB:", round(archive.stat().st_size / 1024**2, 2))

with zipfile.ZipFile(archive) as z:
    bad = z.testzip()

    if bad is not None:
        raise ValueError(f"File ZIP lỗi tại: {bad}")

    total_size = 0
    root = CABBAGE_DIR.resolve()

    for member in z.infolist():
        target = (root / member.filename).resolve()

        if not target.is_relative_to(root):
            raise ValueError("ZIP có đường dẫn không an toàn")

        total_size += member.file_size

    if total_size > 5 * 1024**3:
        raise RuntimeError("Dung lượng giải nén vượt 5 GB")

    if shutil.disk_usage(CABBAGE_DIR).free < total_size * 1.15:
        raise RuntimeError("Không đủ dung lượng ổ đĩa")

    # Tránh ghi đè dữ liệu đã tồn tại
    if any(CABBAGE_DIR.iterdir()):
        raise RuntimeError(
            "Thư mục cabbage đã có dữ liệu. "
            "Hãy kiểm tra trước khi giải nén."
        )

    z.extractall(CABBAGE_DIR)

print("ĐÃ GIẢI NÉN THÀNH CÔNG")
print("Thư mục:", CABBAGE_DIR)

In [ ]:
import yaml
from collections import Counter

yaml_files = list(CABBAGE_DIR.rglob("data.yaml"))

print("Số file data.yaml:", len(yaml_files))

if len(yaml_files) != 1:
    raise RuntimeError(
        f"Cần đúng một data.yaml, tìm thấy {len(yaml_files)}: {yaml_files}"
    )

yaml_path = yaml_files[0]
YOLO_ROOT = yaml_path.parent

with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

names = config["names"]

if isinstance(names, dict):
    class_names = {
        int(k): str(v) for k, v in names.items()
    }
else:
    class_names = dict(enumerate(names))

print("\nDANH SÁCH NHÃN:")
for class_id, name in sorted(class_names.items()):
    print(f"{class_id}: {name}")

print("\nCẤU TRÚC DATASET:")

for split in ["train", "valid", "test"]:
    images_dir = YOLO_ROOT / split / "images"
    labels_dir = YOLO_ROOT / split / "labels"

    image_count = (
        len(list(images_dir.iterdir()))
        if images_dir.exists() else 0
    )

    label_count = (
        len(list(labels_dir.glob("*.txt")))
        if labels_dir.exists() else 0
    )

    print(
        f"{split}: {image_count} ảnh, "
        f"{label_count} file nhãn"
    )

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
import random

images_dir = YOLO_ROOT / "train" / "images"
labels_dir = YOLO_ROOT / "train" / "labels"

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

image_files = [
    p for p in images_dir.iterdir()
    if p.suffix.lower() in EXTENSIONS
    and (labels_dir / f"{p.stem}.txt").exists()
]

rng = random.Random(42)

def preview_yolo(image_path):
    label_path = labels_dir / f"{image_path.stem}.txt"

    with Image.open(image_path) as img:
        preview = img.convert("RGB")

    width, height = preview.size
    draw = ImageDraw.Draw(preview)

    for line in label_path.read_text(
        encoding="utf-8"
    ).splitlines():

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:])

        x1 = (xc - bw / 2) * width
        y1 = (yc - bh / 2) * height
        x2 = (xc + bw / 2) * width
        y2 = (yc + bh / 2) * height

        draw.rectangle(
            [x1, y1, x2, y2],
            outline="red",
            width=3
        )

        draw.text(
            (max(0, x1), max(0, y1 - 15)),
            class_names.get(class_id, str(class_id)),
            fill="red"
        )

    preview.thumbnail((500, 500))
    display(preview)

for image_path in rng.sample(
    image_files, min(5, len(image_files))
):
    print("Ảnh:", image_path.name)
    preview_yolo(image_path)

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import pandas as pd
import hashlib
import json

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

CROP_DIR = (
    PROJECT_DIR / "datasets"
    / "state_crop_candidates"
    / "cabbage"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

# Mapping xác nhận từ data.yaml
LABEL_MAP = {
    0: "cabbage__normal_appearance",
    2: "cabbage__visible_defect"
}

# Bỏ qua moderate fresh (ID 1)
MIN_CROP_SIZE = 48
PADDING_RATIO = 0.05

EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp"
}

assert class_names[0].strip().lower() == "fresh"
assert class_names[1].strip().lower() == "moderate fresh"
assert class_names[2].strip().lower() == "rotten"

print("Thư mục YOLO:", YOLO_ROOT)
print("Thư mục crop:", CROP_DIR)
print("Nhãn được sử dụng:", LABEL_MAP)

In [ ]:
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# Ngăn trộn với lần xuất trước
if any(CROP_DIR.rglob("*.jpg")):
    raise RuntimeError(
        "Thư mục crop đã có ảnh. "
        "Hãy kiểm tra trước khi chạy lại."
    )

crop_records = []
crop_errors = []
skipped = []

for source_split in ["train", "valid", "test"]:

    images_dir = YOLO_ROOT / source_split / "images"
    labels_dir = YOLO_ROOT / source_split / "labels"

    image_paths = [
        p for p in images_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in EXTENSIONS
    ]

    for image_path in tqdm(
        image_paths,
        desc=f"Cropping {source_split}"
    ):
        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            skipped.append({
                "path": str(image_path),
                "reason": "missing_annotation"
            })
            continue

        try:
            with Image.open(image_path) as im:
                image = ImageOps.exif_transpose(im).convert("RGB")

            width, height = image.size

            lines = label_path.read_text(
                encoding="utf-8"
            ).splitlines()

            for box_index, line in enumerate(lines):
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                class_id = int(parts[0])

                if class_id not in LABEL_MAP:
                    continue

                xc, yc, bw, bh = map(float, parts[1:])

                if not (
                    0 <= xc <= 1 and 0 <= yc <= 1
                    and 0 < bw <= 1 and 0 < bh <= 1
                ):
                    skipped.append({
                        "path": str(image_path),
                        "reason": "invalid_box"
                    })
                    continue

                x1 = (xc - bw / 2) * width
                y1 = (yc - bh / 2) * height
                x2 = (xc + bw / 2) * width
                y2 = (yc + bh / 2) * height

                pad_x = bw * width * PADDING_RATIO
                pad_y = bh * height * PADDING_RATIO

                x1 = max(0, int(x1 - pad_x))
                y1 = max(0, int(y1 - pad_y))
                x2 = min(width, int(x2 + pad_x))
                y2 = min(height, int(y2 + pad_y))

                if (
                    x2 - x1 < MIN_CROP_SIZE
                    or y2 - y1 < MIN_CROP_SIZE
                ):
                    skipped.append({
                        "path": str(image_path),
                        "reason": "crop_too_small"
                    })
                    continue

                crop = image.crop((x1, y1, x2, y2))

                target_class = LABEL_MAP[class_id]
                target_dir = CROP_DIR / target_class

                target_dir.mkdir(
                    parents=True,
                    exist_ok=True
                )

                uid = hashlib.sha256(
                    f"{source_split}/{image_path.name}/{box_index}"
                    .encode("utf-8")
                ).hexdigest()[:20]

                target_path = target_dir / f"{uid}.jpg"

                crop.save(
                    target_path,
                    format="JPEG",
                    quality=95
                )

                crop_records.append({
                    "source_path": str(image_path),
                    "source_split": source_split,
                    "label_path": str(label_path),
                    "box_index": box_index,
                    "class_id": class_id,
                    "target_class": target_class,
                    "crop_path": str(target_path),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                })

        except Exception as error:
            crop_errors.append({
                "image_path": str(image_path),
                "error": str(error)
            })

df_crops = pd.DataFrame(crop_records)

df_crops.to_csv(
    REPORT_DIR / "cabbage_crop_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(skipped).to_csv(
    REPORT_DIR / "cabbage_crop_skipped.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(crop_errors).to_csv(
    REPORT_DIR / "cabbage_crop_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN THÀNH CROP")
print("Ảnh crop:", len(df_crops))
print("Lỗi:", len(crop_errors))
print("Bản ghi bỏ qua:", len(skipped))

display(
    df_crops.groupby("target_class")
    .size()
    .reset_index(name="images")
)

In [ ]:
from PIL import Image
from IPython.display import display
import random

rng = random.Random(42)

for class_name, part in df_crops.groupby("target_class"):
    print("\n===== ", class_name, " =====")

    samples = part.sample(
        min(5, len(part)),
        random_state=42
    )

    for path in samples["crop_path"]:
        with Image.open(path) as im:
            preview = im.convert("RGB")
            preview.thumbnail((250, 250))
            display(preview)

In [ ]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

df_crops["sha256"] = [
    sha256_file(path)
    for path in tqdm(df_crops["crop_path"], desc="Hash crops")
]

print(
    "Số bản dư trùng SHA256:",
    df_crops.duplicated("sha256").sum()
)

conflicts = (
    df_crops.groupby("sha256")["target_class"]
    .nunique()
)

conflict_hashes = set(
    conflicts[conflicts > 1].index
)

print("Nhóm ảnh trùng nhưng khác nhãn:", len(conflict_hashes))

df_crops.to_csv(
    REPORT_DIR / "cabbage_crop_manifest_hashed.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image
import imagehash
from tqdm.auto import tqdm

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

manifest_path = REPORT_DIR / "cabbage_crop_manifest_hashed.csv"

df_cabbage = pd.read_csv(manifest_path)

print("Tổng crop:", len(df_cabbage))
print("Danh sách nhãn:")
print(df_cabbage["target_class"].value_counts())

In [ ]:
import re

def video_group(filename):
    name = Path(filename).stem.lower()

    # Bỏ hậu tố Roboflow
    name = re.sub(r"\.rf\.[a-z0-9]+$", "", name)

    # Nhận diện dạng Rotten_mp4-0063_jpg
    match = re.search(r"^(.+?mp4)[-_]\d+", name)

    if match:
        return "video:" + match.group(1)

    return None

df_cabbage["phash"] = [
    str(imagehash.phash(
        Image.open(p).convert("RGB")
    ))
    for p in tqdm(df_cabbage["crop_path"], desc="Calculating pHash")
]

df_cabbage["video_group"] = df_cabbage["source_path"].apply(
    video_group
)

print("Số crop thuộc nhóm video nhận diện được:",
      df_cabbage["video_group"].notna().sum())

display(
    df_cabbage[
        df_cabbage["video_group"].notna()
    ][["source_path", "video_group"]].head(10)
)

df_cabbage.to_csv(
    REPORT_DIR / "cabbage_phash_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import numpy as np
from collections import defaultdict

df_cabbage = df_cabbage.reset_index(drop=True)

parent = list(range(len(df_cabbage)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(a, b):
    a, b = find(a), find(b)
    if a != b:
        parent[b] = a

seen_source = {}
seen_phash = {}
seen_video = {}

for i, row in enumerate(df_cabbage.to_dict("records")):

    source = row["source_path"]

    if source in seen_source:
        union(i, seen_source[source])
    else:
        seen_source[source] = i

    h = row["phash"]
    if h in seen_phash:
        union(i, seen_phash[h])
    else:
        seen_phash[h] = i

    video = row["video_group"]
    if pd.notna(video):
        key = (row["target_class"], video)
        if key in seen_video:
            union(i, seen_video[key])
        else:
            seen_video[key] = i

df_cabbage["group_id"] = [
    find(i) for i in range(len(df_cabbage))
]

group_classes = (
    df_cabbage.groupby("group_id")["target_class"].nunique()
)

conflict_groups = set(
    group_classes[group_classes > 1].index
)

print("Tổng nhóm:", df_cabbage["group_id"].nunique())
print("Nhóm chứa nhiều nhãn:", len(conflict_groups))

df_review_groups = df_cabbage[
    df_cabbage["group_id"].isin(conflict_groups)
]

df_review_groups.to_csv(
    REPORT_DIR / "cabbage_group_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Kiểm tra toàn bộ ảnh trong từng nhóm
group_summary = (
    df_cabbage.groupby("group_id")
    .agg(
        images=("crop_path", "count"),
        classes=("target_class", "nunique"),
        sources=("source_path", "nunique")
    )
    .reset_index()
)

print("Tổng nhóm:", len(group_summary))
print("Nhóm lớn nhất:", group_summary["images"].max())

print("\n10 nhóm có nhiều ảnh nhất:")
display(
    group_summary.sort_values(
        "images", ascending=False
    ).head(10)
)

print("\nSố nhóm theo từng lớp:")

display(
    df_cabbage.groupby("target_class")["group_id"]
    .nunique()
    .reset_index(name="groups")
)

In [ ]:
video_summary = (
    df_cabbage.dropna(subset=["video_group"])
    .groupby("video_group")
    .agg(
        images=("crop_path", "count"),
        labels=("target_class", "nunique"),
        source_images=("source_path", "nunique")
    )
    .reset_index()
    .sort_values("images", ascending=False)
)

display(video_summary.head(20))

print(
    "Số video/nhóm tên video:",
    len(video_summary)
)

In [ ]:
from PIL import Image
from IPython.display import display

largest_groups = (
    group_summary.sort_values(
        "images", ascending=False
    )["group_id"]
    .head(3)
    .tolist()
)

for gid in largest_groups:
    part = df_cabbage[
        df_cabbage["group_id"] == gid
    ]

    print("\n" + "=" * 45)
    print("Group ID:", gid)
    print("Số ảnh:", len(part))
    print("Nhãn:", part["target_class"].unique())

    samples = part.sample(
        min(4, len(part)),
        random_state=42
    )

    for path in samples["crop_path"]:
        with Image.open(path) as im:
            preview = im.convert("RGB")
            preview.thumbnail((240, 240))
            display(preview)

In [ ]:
import pandas as pd
from pathlib import Path

print("TỔNG SỐ VIDEO NHẬN DIỆN")
display(
    df_cabbage.groupby(
        ["video_group", "target_class"]
    ).agg(
        images=("crop_path", "count"),
        source_images=("source_path", "nunique")
    )
)

print("\nMẪU TÊN FILE NGUỒN")

for video, part in df_cabbage.groupby("video_group"):
    print("\nVIDEO:", video)

    for path in part["source_path"].head(5):
        print(Path(path).name)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd
from tqdm.auto import tqdm

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

CABBAGE_TRAIN_DIR = (
    PROJECT_DIR
    / "datasets"
    / "efficientnet_v4_cabbage_candidates"
    / "train"
)

if CABBAGE_TRAIN_DIR.exists() and any(
    p.is_file() for p in CABBAGE_TRAIN_DIR.rglob("*")
):
    raise RuntimeError(
        "Thư mục đích đã có dữ liệu. Không ghi đè."
    )

records = []

for row in tqdm(
    df_cabbage.to_dict("records"),
    desc="Export cabbage candidates"
):
    source = Path(row["crop_path"])

    target_dir = (
        CABBAGE_TRAIN_DIR / row["target_class"]
    )

    target_dir.mkdir(parents=True, exist_ok=True)

    target = target_dir / source.name

    shutil.copy2(source, target)

    records.append({
        "source": str(source),
        "target": str(target),
        "target_class": row["target_class"],
        "video_group": row["video_group"],
        "split": "train_candidate"
    })

df_export_cabbage = pd.DataFrame(records)

REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

df_export_cabbage.to_csv(
    REPORT_DIR / "cabbage_train_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tổng ảnh:", len(df_export_cabbage))
print("Số lớp:", df_export_cabbage["target_class"].nunique())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

BULK_ZIP_DIR = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
    / "archives"
)

BULK_ZIP_DIR.mkdir(parents=True, exist_ok=True)

print("Hãy lưu 5 ZIP tại:")
print(BULK_ZIP_DIR)


In [ ]:
%run scripts/bulk_state_dataset.py inspect

In [ ]:
%run scripts/bulk_state_dataset.py prepare

In [ ]:
%run scripts/bulk_state_dataset.py audit

In [ ]:
%run scripts/bulk_state_dataset.py summary

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

ARCHIVE_DIR = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
    / "archives"
)

print("Thư mục:", ARCHIVE_DIR)
print("Tồn tại:", ARCHIVE_DIR.exists())

print("\n===== DANH SÁCH FILE =====")

for path in ARCHIVE_DIR.iterdir():
    print(
        path.name,
        "|",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

In [ ]:
import zipfile

required = [
    "lettuce.zip",
    "smartveggies.zip",
    "onion.zip",
    "pork.zip",
    "shrimp.zip"
]

all_ready = True

for name in required:
    path = ARCHIVE_DIR / name

    if not path.exists():
        print("THIẾU:", name)
        all_ready = False
        continue

    try:
        with zipfile.ZipFile(path) as z:
            if z.testzip() is not None:
                print("ZIP LỖI:", name)
                all_ready = False
            else:
                print(
                    "OK:",
                    name,
                    round(path.stat().st_size / 1024**2, 2),
                    "MB"
                )
    except zipfile.BadZipFile:
        print("KHÔNG PHẢI ZIP HỢP LỆ:", name)
        all_ready = False

print("\nSẵn sàng xử lý:", all_ready)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

ARCHIVE_DIR = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
    / "archives"
)

ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [
    PROJECT_DIR / "datasets" / "raw",
    Path.home() / "Downloads",
    Path.home() / "Desktop",
]

keywords = [
    "lettuce",
    "smartveggies",
    "detectvagetable",
    "onion",
    "meat-checker",
    "pork",
    "shrimp"
]

found = set()

for base in SEARCH_DIRS:
    if not base.exists():
        continue

    for path in base.rglob("*.zip"):
        name = path.name.lower()

        if any(k in name for k in keywords):
            found.add(path)

print("===== DATASET ZIP TÌM ĐƯỢC =====")

for path in sorted(found):
    print(
        path,
        "|",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nTổng ZIP tìm được:", len(found))

In [ ]:
import shutil
import zipfile

name_map = {
    "lettuce": "lettuce.zip",
    "smartveggies": "smartveggies.zip",
    "detectvagetable": "smartveggies.zip",
    "onion": "onion.zip",
    "meat-checker": "pork.zip",
    "pork": "pork.zip",
    "shrimp": "shrimp.zip",
}

for source in sorted(found):
    name = source.name.lower()

    matches = [
        target for keyword, target in name_map.items()
        if keyword in name
    ]

    matches = set(matches)

    if len(matches) != 1:
        print("Bỏ qua, cần kiểm tra thủ công:", source.name)
        continue

    target = ARCHIVE_DIR / matches.pop()

    if target.exists():
        print("Đã tồn tại, không ghi đè:", target.name)
        continue

    if not zipfile.is_zipfile(source):
        print("Không phải ZIP hợp lệ:", source.name)
        continue

    shutil.copy2(source, target)

    print("Đã sao chép:", source.name, "→", target.name)

In [ ]:
import zipfile

required = [
    "lettuce.zip",
    "smartveggies.zip",
    "onion.zip",
    "pork.zip",
    "shrimp.zip"
]

for name in required:
    path = ARCHIVE_DIR / name

    if not path.exists():
        print("THIẾU:", name)
    elif zipfile.is_zipfile(path):
        print("OK:", name)
    else:
        print("ZIP KHÔNG HỢP LỆ:", name)

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%run scripts/bulk_state_dataset.py inspect

In [ ]:
from pathlib import Path
from collections import defaultdict
import zipfile

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

ARCHIVE_DIR = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4" / "archives"
)

lettuce_zip = ARCHIVE_DIR / "lettuce.zip"

with zipfile.ZipFile(lettuce_zip) as z:
    groups = defaultdict(list)

    for member in z.infolist():
        if member.is_dir():
            continue

        normalized = member.filename.replace("\\", "/").casefold()
        groups[normalized].append(member.filename)

    duplicates = {
        key: values
        for key, values in groups.items()
        if len(values) > 1
    }

    print("Tổng file ZIP:", len(z.infolist()))
    print("Số đường dẫn bị trùng:", len(duplicates))

    for key, values in list(duplicates.items())[:20]:
        print("\nTrùng:", key)
        for value in values:
            print(" -", value)

    print("\nCác data.yaml:")
    for member in z.infolist():
        if member.filename.lower().endswith("data.yaml"):
            print(member.filename)

In [ ]:
import zipfile
import hashlib

lettuce_zip = ARCHIVE_DIR / "lettuce.zip"

with zipfile.ZipFile(lettuce_zip) as z:
    yaml_files = [
        m for m in z.infolist()
        if m.filename == "data.yaml"
    ]

    for i, member in enumerate(yaml_files, 1):
        content = z.read(member)

        print(f"\n===== DATA.YAML {i} =====")
        print("SHA256:", hashlib.sha256(content).hexdigest())
        print(content.decode("utf-8", errors="replace"))

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from collections import Counter
import hashlib

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

ARCHIVE_DIR = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
    / "archives"
)

source_zip = ARCHIVE_DIR / "lettuce.zip"
clean_zip = ARCHIVE_DIR / "lettuce_clean.zip"

if clean_zip.exists():
    raise FileExistsError(
        f"File đã tồn tại: {clean_zip}"
    )

seen = {}
removed = 0

with ZipFile(source_zip, "r") as src:
    with ZipFile(clean_zip, "w", ZIP_DEFLATED) as dst:
        for member in src.infolist():

            if member.is_dir():
                continue

            name = member.filename.replace("\\", "/")
            parts = name.split("/")

            if (
                name.startswith("/")
                or ".." in parts
                or ":" in parts[0]
            ):
                raise ValueError(
                    f"Đường dẫn không an toàn: {name}"
                )

            key = name.casefold()
            content = src.read(member)
            digest = hashlib.sha256(content).hexdigest()

            if key in seen:
                if seen[key] != digest:
                    raise RuntimeError(
                        f"Trùng tên nhưng khác nội dung: {name}"
                    )

                removed += 1
                continue

            seen[key] = digest
            dst.writestr(name, content)

print("ZIP mới:", clean_zip)
print("File trùng đã bỏ:", removed)

with ZipFile(clean_zip) as z:
    print("Kiểm tra ZIP:", z.testzip())

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%run scripts/bulk_state_dataset.py inspect

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%run scripts/bulk_state_dataset.py inspect

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%pip install pillow pyyaml
%run scripts/state_v4_review_local.py

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter
from zipfile import ZipFile
import csv
import io

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
SHRIMP_ZIP = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
    / "archives" / "shrimp.zip"
)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

with ZipFile(SHRIMP_ZIP) as z:
    files = [m for m in z.infolist() if not m.is_dir()]

    images = [
        m for m in files
        if PurePosixPath(m.filename).suffix.lower() in IMAGE_EXTS
    ]

    print("===== 1. THỐNG KÊ ZIP =====")
    print("Tổng file:", len(files))
    print("Tổng file ảnh:", len(images))

    print("\n===== 2. CẤU TRÚC ĐƯỜNG DẪN ẢNH =====")
    for m in images[:20]:
        print(m.filename)

    print("\n===== 3. THƯ MỤC CHỨA ẢNH =====")
    folder_counts = Counter(
        str(PurePosixPath(m.filename).parent)
        for m in images
    )
    for folder, count in folder_counts.most_common(20):
        print(folder, ":", count)

    print("\n===== 4. FILE CHỨA NHÃN =====")
    metadata = [
        m for m in files
        if PurePosixPath(m.filename).suffix.lower()
        in {".csv", ".yaml", ".yml", ".json"}
    ]

    for m in metadata[:30]:
        print(m.filename)

    print("\n===== 5. KIỂM TRA FILE CSV =====")

    for m in metadata:
        if PurePosixPath(m.filename).suffix.lower() != ".csv":
            continue

        print("\nFILE:", m.filename)

        text_data = z.read(m).decode(
            "utf-8-sig", errors="replace"
        )

        reader = csv.DictReader(io.StringIO(text_data))
        rows = list(reader)

        print("Cột:", reader.fieldnames)
        print("Số dòng:", len(rows))
        print("5 dòng đầu:")

        for row in rows[:5]:
            print(row)

        # Nếu có cột one-hot, đếm các giá trị 1
        for column in reader.fieldnames or []:
            values = [str(r.get(column, "")).strip() for r in rows]

            if values and set(values).issubset({"0", "1", ""}):
                print(
                    "Nhãn", column,
                    "| positive:", values.count("1")
                )

    print("\n===== 6. MẪU FILE TXT =====")

    txt_files = [
        m for m in files
        if PurePosixPath(m.filename).suffix.lower() == ".txt"
    ]

    for m in txt_files[:5]:
        print("\n", m.filename)
        print(
            z.read(m).decode(
                "utf-8-sig", errors="replace"
            )[:300]
        )

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

REVIEW_DIR = (
    PROJECT_DIR / "datasets"
    / "reports" / "state_v4" / "review"
)

preview_files = [
    "pork_kasim_fresh.jpg",
    "pork_kasim_spoiled.jpg",
    "pork_liempo_fresh.jpg",
    "pork_liempo_spoiled.jpg",
]

for name in preview_files:
    path = REVIEW_DIR / name

    if path.exists():
        print("\n===== ", name, " =====")
        display(Image.open(path))
    else:
        print("Không tìm thấy:", name)

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import io

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

SHRIMP_ZIP = (
    PROJECT_DIR / "datasets" / "raw"
    / "state_sources" / "extension_v4"
    / "archives" / "shrimp.zip"
)

with ZipFile(SHRIMP_ZIP) as z:
    csv_content = z.read("valid/_classes.csv")

df_shrimp = pd.read_csv(io.BytesIO(csv_content))

# Loại khoảng trắng khỏi tên cột
df_shrimp.columns = df_shrimp.columns.str.strip()

for col in ["fresh", "not"]:
    df_shrimp[col] = pd.to_numeric(
        df_shrimp[col], errors="raise"
    )

print("Tổng dòng CSV:", len(df_shrimp))
print("Số tên file duy nhất:", df_shrimp["filename"].nunique())

print("\nPHÂN BỐ KẾT HỢP:")
display(
    df_shrimp.groupby(["fresh", "not"])
    .size()
    .reset_index(name="images")
)

print("\nGIÁ TRỊ NHÃN:")
print(df_shrimp[["fresh", "not"]].sum())

In [ ]:
import pandas as pd

ISSUES_PATH = (
    PROJECT_DIR / "datasets"
    / "reports" / "state_v4"
    / "inspect_issues.csv"
)

if ISSUES_PATH.exists():
    issues = pd.read_csv(ISSUES_PATH)

    mask = issues.astype(str).apply(
        lambda col: col.str.contains(
            "smartveggies",
            case=False,
            na=False
        )
    ).any(axis=1)

    smart_issues = issues[mask]

    print("Số cảnh báo SmartVeggies:", len(smart_issues))
    display(smart_issues)

else:
    print("Không tìm thấy inspect_issues.csv")

In [ ]:
from pathlib import Path

CONFIG_PATH = (
    PROJECT_DIR / "config"
    / "state_sources_v4.yaml"
)

print(CONFIG_PATH.read_text(encoding="utf-8"))

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%run scripts/bulk_state_dataset.py prepare

In [ ]:
%run scripts/bulk_state_dataset.py audit
%run scripts/bulk_state_dataset.py summary

In [ ]:
REPORT_DIR = (
    PROJECT_DIR / "datasets"
    / "reports" / "state_v4"
)

print("CÁC BÁO CÁO V4:")

for p in sorted(REPORT_DIR.rglob("*")):
    if p.is_file():
        print(
            p.relative_to(REPORT_DIR),
            "|",
            round(p.stat().st_size / 1024, 1),
            "KB"
        )

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

REPORT_DIR = (
    PROJECT_DIR / "datasets"
    / "reports" / "state_v4"
)

for name in [
    "summary.json",
    "audit_state.json",
    "prepare_state.json"
]:
    path = REPORT_DIR / name

    print("\n" + "=" * 60)
    print(name)

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(json.dumps(
            data,
            ensure_ascii=False,
            indent=2
        )[:12000])
    else:
        print("Không tìm thấy file.")

In [ ]:
files = {
    "manifest": "manifest.csv",
    "prepare_review": "prepare_review.csv",
    "split_plan": "split_plan.csv",
    "audit_issues": "audit_issues.csv",
}

for label, filename in files.items():

    path = REPORT_DIR / filename

    print("\n" + "=" * 60)
    print(label.upper())

    if not path.exists():
        print("Không tìm thấy:", filename)
        continue

    df = pd.read_csv(path, low_memory=False)

    print("Số dòng:", len(df))
    print("Danh sách cột:")
    print(df.columns.tolist())

    display(df.head(3))

    # Tìm những cột trạng thái hoặc lý do kiểm duyệt
    relevant_cols = [
        c for c in df.columns
        if any(
            keyword in c.lower()
            for keyword in [
                "status",
                "reason",
                "decision",
                "issue",
                "review",
                "split"
            ]
        )
    ]

    for col in relevant_cols[:8]:
        print(f"\nPhân bố cột: {col}")
        display(
            df[col]
            .fillna("(trống)")
            .astype(str)
            .value_counts()
            .head(15)
            .to_frame("count")
        )

In [ ]:
class_path = REPORT_DIR / "class_summary.csv"

df_class = pd.read_csv(class_path)

print("SỐ LỚP TRONG BÁO CÁO:", len(df_class))
print("CÁC CỘT:", df_class.columns.tolist())

display(df_class)

In [ ]:
for filename in [
    "duplicate_groups.csv",
    "audit_pairs.csv",
    "audit_issues.csv"
]:
    path = REPORT_DIR / filename

    print("\n" + "=" * 60)
    print(filename)

    if not path.exists():
        continue

    df = pd.read_csv(path, low_memory=False)

    print("Số dòng:", len(df))
    print("Các cột:", df.columns.tolist())

    display(df.head(10))

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "manifest.csv",
    low_memory=False
)

print("Tổng provenance records:", len(df))

print("\nPHƯƠNG PHÁP PHÂN NHÓM:")
display(
    df.groupby(
        ["source_dataset", "group_method"],
        dropna=False
    ).agg(
        records=("record_id", "count"),
        groups=("image_group", "nunique"),
        source_images=("source_image", "nunique")
    ).reset_index()
)

print("\nNHÓM NGUỒN THEO LỚP:")
display(
    df.groupby("target_class").agg(
        records=("record_id", "count"),
        groups=("image_group", "nunique"),
        source_images=("source_image", "nunique")
    ).reset_index()
)

In [ ]:
pairs = pd.read_csv(
    REPORT_DIR / "audit_pairs.csv"
)

# Chuẩn hóa đường dẫn để so sánh
def normalize_path(value):
    return str(value).replace("\\", "/").casefold()

pairs["same_path"] = [
    normalize_path(a) == normalize_path(b)
    for a, b in zip(pairs["first"], pairs["second"])
]

print("Tổng cặp:", len(pairs))

print("Hai đầu cùng một đường dẫn:",
      pairs["same_path"].sum())

print("Hai đường dẫn khác nhau:",
      (~pairs["same_path"]).sum())

print("\nPHÂN BỐ KIỂU TRÙNG:")
display(
    pairs.groupby(
        ["kind", "same_path"],
        dropna=False
    ).size().reset_index(name="pairs")
)

print("\nCẶP CẦN KIỂM TRA RIÊNG:")
display(
    pairs.loc[
        (~pairs["same_path"])
        | pairs["label_conflict"].astype(str).str.lower().eq("true"),
        [
            "first", "second", "kind", "distance",
            "first_class", "second_class", "label_conflict"
        ]
    ].head(20)
)

In [ ]:
from PIL import Image
from collections import Counter

candidate_root = (
    PROJECT_DIR / "datasets" / "state_v4_candidates"
)

image_exts = {".jpg", ".jpeg", ".png", ".webp"}

all_images = [
    p for p in candidate_root.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
]

counts = Counter(p.parent.name for p in all_images)

print("Tổng file ảnh tồn tại:", len(all_images))
print("\nSố ảnh theo lớp:")

for class_name, count in sorted(counts.items()):
    print(f"{class_name}: {count}")

# Kiểm tra đọc được toàn bộ ảnh
bad_images = []

for path in all_images:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as error:
        bad_images.append((str(path), str(error)))

print("\nẢnh không đọc được:", len(bad_images))

for item in bad_images[:10]:
    print(item)

In [ ]:
import json

class_summary = pd.read_csv(
    REPORT_DIR / "class_summary.csv"
)

existing_counts = {
    p.name: sum(
        1 for f in p.iterdir()
        if f.is_file() and f.suffix.lower() in image_exts
    )
    for p in candidate_root.iterdir()
    if p.is_dir()
}

class_summary["files_on_disk"] = (
    class_summary["target_class"]
    .map(existing_counts)
    .fillna(0)
    .astype(int)
)

class_summary["ready_for_merge"] = False

class_summary["next_action"] = class_summary.apply(
    lambda row:
        "Collect missing labeled data"
        if row["unique_images"] == 0
        else "Review source grouping and image quality",
    axis=1
)

output_path = REPORT_DIR / "v4_merge_readiness.csv"

class_summary.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

display(class_summary)

print("\nĐã lưu:", output_path)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"
CANDIDATE_DIR = PROJECT_DIR / "datasets" / "state_v4_candidates"

manifest = pd.read_csv(
    REPORT_DIR / "manifest.csv",
    low_memory=False
)

def resolve_candidate_path(value):
    path = Path(str(value))

    if path.is_absolute():
        return path.resolve()

    return (PROJECT_DIR / path).resolve()

manifest["resolved_path"] = (
    manifest["crop_path"].apply(resolve_candidate_path)
)

referenced_paths = set(manifest["resolved_path"])
disk_paths = {
    p.resolve()
    for p in CANDIDATE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower()
    in {".jpg", ".jpeg", ".png", ".webp"}
}

missing_paths = referenced_paths - disk_paths
unreferenced_paths = disk_paths - referenced_paths

print("===== KIỂM TRA ĐỐI CHIẾU =====")
print("Records trong manifest:", len(manifest))
print("Đường dẫn duy nhất:", len(referenced_paths))
print("Ảnh thực tế:", len(disk_paths))
print("Ảnh manifest không tìm thấy:", len(missing_paths))
print("Ảnh không thuộc manifest:", len(unreferenced_paths))

print("\nẢNH KHÔNG THUỘC MANIFEST (10 FILE ĐẦU):")
for p in sorted(unreferenced_paths)[:10]:
    print(p)

pd.DataFrame({
    "path": [str(p) for p in sorted(unreferenced_paths)]
}).to_csv(
    REPORT_DIR / "unreferenced_on_disk.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import hashlib
from tqdm.auto import tqdm

unique_manifest = manifest.drop_duplicates(
    subset=["resolved_path"]
).copy()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if missing_paths:
    print("DỪNG: Còn đường dẫn thiếu, chưa kiểm tra hash.")
else:
    unique_manifest["actual_sha256"] = [
        sha256_file(p)
        for p in tqdm(
            unique_manifest["resolved_path"],
            desc="Checking SHA256"
        )
    ]

    # So sánh bản ghi hiện tại với hash thực tế
    unique_manifest["hash_matches_manifest"] = (
        unique_manifest["sha256"].astype(str).str.lower()
        == unique_manifest["actual_sha256"]
    )

    print(
        "Hash không khớp manifest:",
        (~unique_manifest["hash_matches_manifest"]).sum()
    )

    # Ảnh có cùng hash nhưng thuộc những lớp khác nhau
    class_conflicts = (
        unique_manifest
        .groupby("actual_sha256")["target_class"]
        .nunique()
    )

    print(
        "Nhóm SHA256 có nhãn mâu thuẫn:",
        (class_conflicts > 1).sum()
    )

    unique_manifest.to_csv(
        REPORT_DIR / "v4_verified_manifest.csv",
        index=False,
        encoding="utf-8-sig"
    )

In [ ]:
import shutil
from pathlib import Path

CLEAN_DIR = (
    PROJECT_DIR / "datasets"
    / "state_v4_current_candidates"
)

if missing_paths:
    raise RuntimeError("Manifest còn tham chiếu ảnh không tồn tại.")

if (class_conflicts > 1).any():
    raise RuntimeError("Còn nhóm SHA256 có nhãn mâu thuẫn.")

if CLEAN_DIR.exists() and any(CLEAN_DIR.iterdir()):
    raise RuntimeError(
        f"Thư mục đích đã có dữ liệu: {CLEAN_DIR}"
    )

# Kiểm tra một đường dẫn không bị gán hai lớp
path_conflicts = (
    manifest.groupby("resolved_path")["target_class"]
    .nunique()
)

if (path_conflicts > 1).any():
    raise RuntimeError(
        "Có cùng đường dẫn xuất hiện với nhiều nhãn."
    )

# Một ảnh thực tế cho mỗi SHA256 và lớp
export_df = unique_manifest.drop_duplicates(
    subset=["actual_sha256", "target_class"]
).copy()

export_records = []

for row in tqdm(
    export_df.to_dict("records"),
    desc="Building V4 candidates"
):
    source = Path(row["resolved_path"])

    target_dir = CLEAN_DIR / row["target_class"]
    target_dir.mkdir(parents=True, exist_ok=True)

    target = target_dir / (
        row["actual_sha256"] + source.suffix.lower()
    )

    shutil.copy2(source, target)

    export_records.append({
        "source": str(source),
        "target": str(target),
        "target_class": row["target_class"],
        "sha256": row["actual_sha256"],
        "status": "review"
    })

export_report = pd.DataFrame(export_records)

export_report.to_csv(
    REPORT_DIR / "v4_current_candidates_export.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTổng ảnh đã xuất:", len(export_report))
print("\nSố ảnh theo lớp:")
display(
    export_report["target_class"]
    .value_counts()
    .rename_axis("target_class")
    .reset_index(name="images")
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

REPORT_DIR = (
    PROJECT_DIR / "datasets" / "reports" / "state_v4"
)

df = pd.read_csv(
    REPORT_DIR / "manifest.csv",
    low_memory=False
)

print("Tổng bản ghi:", len(df))

print("\n1. PHƯƠNG PHÁP NHÓM NGUỒN")
display(
    df.groupby(
        ["source_dataset", "group_method"],
        dropna=False
    ).agg(
        records=("record_id", "count"),
        groups=("image_group", "nunique"),
        source_images=("source_image", "nunique")
    ).reset_index()
)

print("\n2. NHÓM NGUỒN THEO LỚP")
display(
    df.groupby("target_class").agg(
        records=("record_id", "count"),
        groups=("image_group", "nunique"),
        source_images=("source_image", "nunique")
    ).reset_index()
)

print("\n3. VÍ DỤ GROUP ID")
display(
    df[
        ["source_dataset", "original_label",
         "source_image", "image_group", "group_method"]
    ].head(15)
)

In [ ]:
import re
from pathlib import PurePosixPath

def get_original_stem(path):
    name = PurePosixPath(
        str(path).replace("\\", "/")
    ).name

    name = re.sub(
        r"\.rf\.[a-fA-F0-9]+(?=\.[^.]+$)",
        "",
        name
    )

    return name

df["original_stem"] = df["source_image"].apply(
    get_original_stem
)

def detect_video(name):
    match = re.search(
        r"^(.+?_mp4)[-_]\d+",
        name.lower()
    )

    return match.group(1) if match else None

df["detected_video"] = df["original_stem"].apply(
    detect_video
)

print("SỐ ẢNH NHẬN DIỆN ĐƯỢC NGUỒN VIDEO")

display(
    df.groupby("source_dataset").agg(
        records=("record_id", "count"),
        detected_video_images=(
            "detected_video",
            lambda x: x.notna().sum()
        ),
        video_names=("detected_video", "nunique")
    ).reset_index()
)

print("\n20 TÊN ẢNH NGUỒN ĐẦU TIÊN:")

display(
    df[
        ["source_dataset", "original_stem",
         "detected_video", "image_group"]
    ].head(20)
)

df.to_csv(
    REPORT_DIR / "v4_source_group_diagnostic.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import random
import math

CLEAN_DIR = (
    PROJECT_DIR / "datasets"
    / "state_v4_current_candidates"
)

rng = random.Random(42)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

for class_dir in sorted(CLEAN_DIR.iterdir()):

    if not class_dir.is_dir():
        continue

    images = [
        p for p in class_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTS
    ]

    if not images:
        continue

    samples = rng.sample(
        images,
        min(8, len(images))
    )

    thumb_w = 180
    thumb_h = 180
    columns = 4
    rows = math.ceil(len(samples) / columns)

    canvas = Image.new(
        "RGB",
        (columns * thumb_w, rows * thumb_h),
        "white"
    )

    for i, path in enumerate(samples):
        with Image.open(path) as im:
            preview = ImageOps.fit(
                im.convert("RGB"),
                (thumb_w, thumb_h)
            )

        x = (i % columns) * thumb_w
        y = (i // columns) * thumb_h

        canvas.paste(preview, (x, y))

    print(
        f"\n{class_dir.name} — {len(images)} ảnh"
    )

    display(canvas)

In [ ]:
from PIL import Image
from IPython.display import display

PORK_REVIEW_DIR = (
    PROJECT_DIR / "datasets"
    / "reports" / "state_v4" / "pork_review"
)

for path in sorted(PORK_REVIEW_DIR.glob("*.jpg")):
    print("\n===== ", path.name, " =====")
    display(Image.open(path))

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

PORK_DIR = (
    PROJECT_DIR / "datasets"
    / "reports" / "state_v4" / "pork_review"
)

files = [
    "kasim_fresh.jpg",
    "kasim_spoiled.jpg",
    "liempo_fresh.jpg",
    "liempo_spoiled.jpg",
    "pork_chop_fresh.jpg",
    "pork_chop_spoiled.jpg",
    "tenderloin_fresh.jpg",
    "tenderloin_spoiled.jpg",
]

for filename in files:
    path = PORK_DIR / filename
    print("\n", filename)

    if path.exists():
        with Image.open(path) as im:
            display(im.copy())
    else:
        print("Không tìm thấy file")

In [ ]:
from pathlib import Path
import yaml

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
CONFIG_PATH = PROJECT_DIR / "config" / "state_sources_v4.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

def show_pork(obj, path="root"):
    if isinstance(obj, dict):
        for key, value in obj.items():
            current = f"{path}.{key}"

            if str(key).lower() == "pork":
                print("VỊ TRÍ:", current)
                print(yaml.safe_dump(
                    value, allow_unicode=True, sort_keys=False
                ))
            else:
                show_pork(value, current)

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            show_pork(value, f"{path}[{i}]")

show_pork(config)

In [ ]:
from pathlib import Path
import yaml
import copy
import difflib

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
CONFIG_PATH = PROJECT_DIR / "config" / "state_sources_v4.yaml"

PORK_MAPPING = {
    "Kasim Fresh": "pork__normal_appearance",
    "Kasim Spoiled": "pork__visible_defect",
    "Liempo Fresh": "pork__normal_appearance",
    "Liempo Spoiled": "pork__visible_defect",
    "Pork chop Fresh": "pork__normal_appearance",
    "Pork chop Spoiled": "pork__visible_defect",
    "Tenderloin Fresh": "pork__normal_appearance",
    "Tenderloin Spoiled": "pork__visible_defect",
}

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    original_config = yaml.safe_load(f)

new_config = copy.deepcopy(original_config)

pork_config = new_config["sources"]["pork"]

# Kiểm tra định dạng trước khi sửa
assert isinstance(pork_config["approved_mappings"], dict)

existing = pork_config["approved_mappings"]

if existing:
    raise RuntimeError(
        f"approved_mappings đã có dữ liệu: {existing}. "
        "Cần kiểm tra trước khi sửa."
    )

# Thêm đúng 8 mapping đã được bạn phê duyệt
pork_config["approved_mappings"] = PORK_MAPPING.copy()

# Không thay đổi nhãn Not Pork Meat
assert "Not Pork Meat" not in pork_config["approved_mappings"]

# Lưu ra file xem trước, chưa ghi đè cấu hình thật
STAGED_PATH = CONFIG_PATH.with_name(
    "state_sources_v4_pork_reviewed.yaml"
)

STAGED_PATH.write_text(
    yaml.safe_dump(
        new_config,
        allow_unicode=True,
        sort_keys=False
    ),
    encoding="utf-8"
)

print("Đã tạo bản cấu hình xem trước:", STAGED_PATH)

print("\nMAPPING PORK MỚI:")
print(yaml.safe_dump(
    pork_config["approved_mappings"],
    allow_unicode=True,
    sort_keys=False
))

old_lines = CONFIG_PATH.read_text(
    encoding="utf-8"
).splitlines()

new_lines = STAGED_PATH.read_text(
    encoding="utf-8"
).splitlines()

print("\n===== SO SÁNH CẤU HÌNH =====")

print("\n".join(difflib.unified_diff(
    old_lines,
    new_lines,
    fromfile="original",
    tofile="pork_reviewed",
    lineterm=""
))[:15000])

In [ ]:
import shutil
from datetime import datetime

BACKUP_PATH = CONFIG_PATH.with_name(
    "state_sources_v4_before_pork_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".yaml"
)

with STAGED_PATH.open("r", encoding="utf-8") as f:
    staged = yaml.safe_load(f)

assert staged["sources"]["pork"]["approved_mappings"] == PORK_MAPPING

# Bảo đảm các nguồn khác không thay đổi về cấu trúc dữ liệu
for source_name in original_config["sources"]:
    if source_name != "pork":
        assert (
            staged["sources"][source_name]
            == original_config["sources"][source_name]
        )

shutil.copy2(CONFIG_PATH, BACKUP_PATH)
shutil.copy2(STAGED_PATH, CONFIG_PATH)

print("Đã sao lưu:", BACKUP_PATH)
print("Đã áp dụng cấu hình Pork:", CONFIG_PATH)

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%run scripts/bulk_state_dataset.py inspect

In [ ]:
%run scripts/bulk_state_dataset.py prepare

In [ ]:
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

LOCK_FILE = (
    PROJECT_DIR
    / "datasets"
    / "state_v4_work"
    / ".bulk_state.lock"
)

print("File khóa tồn tại:", LOCK_FILE.exists())

if LOCK_FILE.exists():
    print("\nNỘI DUNG FILE KHÓA:")
    print(
        LOCK_FILE.read_text(
            encoding="utf-8",
            errors="replace"
        )[:3000]
    )

    print("\nThời gian chỉnh sửa cuối:")
    print(
        datetime.fromtimestamp(
            LOCK_FILE.stat().st_mtime
        )
    )

In [ ]:
import subprocess

result = subprocess.run(
    [
        "powershell",
        "-NoProfile",
        "-Command",
        "Get-CimInstance Win32_Process "
        "| Where-Object { $_.Name -match 'python|jupyter' } "
        "| Select-Object ProcessId, Name, CommandLine "
        "| Format-List"
    ],
    capture_output=True,
    text=True,
    errors="replace"
)

print(result.stdout)

In [ ]:
from datetime import datetime

if LOCK_FILE.exists():
    backup_lock = LOCK_FILE.with_name(
        ".bulk_state.stale_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + ".lock"
    )

    LOCK_FILE.rename(backup_lock)

    print("Đã lưu file khóa cũ tại:")
    print(backup_lock)
else:
    print("Không còn file khóa.")

In [ ]:
%cd C:\Users\ASUS\SIC-Food
%run scripts/bulk_state_dataset.py prepare

In [ ]:
%run scripts/bulk_state_dataset.py audit
%run scripts/bulk_state_dataset.py summary

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

REPORT_DIR = (
    PROJECT_DIR / "datasets" / "reports" / "state_v4"
)

df_class = pd.read_csv(
    REPORT_DIR / "class_summary.csv"
)

print("===== TỔNG HỢP DATASET V4 =====")

display(
    df_class[
        [
            "target_class",
            "records",
            "unique_images",
            "independent_audit_groups",
            "review_images",
            "train_candidate",
            "status"
        ]
    ]
)

print("\n===== CÁC LỚP CÒN THIẾU =====")

display(
    df_class[
        df_class["unique_images"] == 0
    ][["target_class", "status"]]
)

print("\n===== PORK =====")

display(
    df_class[
        df_class["ingredient"] == "pork"
    ]
)

In [ ]:
df_manifest = pd.read_csv(
    REPORT_DIR / "manifest.csv",
    low_memory=False
)

print("===== PHƯƠNG PHÁP NHÓM =====")

display(
    df_manifest.groupby(
        ["source_dataset", "group_method"],
        dropna=False
    ).agg(
        records=("record_id", "count"),
        groups=("image_group", "nunique"),
        source_images=("source_image", "nunique")
    ).reset_index()
)

print("\n===== NHÓM THEO LỚP =====")

display(
    df_manifest.groupby("target_class").agg(
        records=("record_id", "count"),
        groups=("image_group", "nunique"),
        sources=("source_image", "nunique")
    ).reset_index()
)

In [ ]:
import shutil
from datetime import datetime

BACKUP_DIR = (
    PROJECT_DIR / "datasets" / "reports"
    / "state_v4_snapshots"
    / "after_pork_10516"
)

if BACKUP_DIR.exists():
    print("Đã có bản sao lưu:", BACKUP_DIR)
else:
    BACKUP_DIR.mkdir(parents=True)

    for name in [
        "manifest.csv",
        "class_summary.csv",
        "split_plan.csv",
        "audit_pairs.csv",
        "audit_issues.csv",
        "summary.json",
        "audit_state.json",
        "prepare_state.json",
    ]:
        source = REPORT_DIR / name

        if source.exists():
            shutil.copy2(source, BACKUP_DIR / name)

    print("Đã sao lưu báo cáo tại:", BACKUP_DIR)

In [ ]:
from pathlib import Path, PurePosixPath
import pandas as pd
import re

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "manifest.csv",
    low_memory=False
)

def original_name(path):
    name = PurePosixPath(
        str(path).replace("\\", "/")
    ).name

    # Bỏ hậu tố nhận dạng do Roboflow sinh ra
    name = re.sub(
        r"\.rf\.[a-fA-F0-9]+(?=\.[^.]+$)",
        "",
        name
    )

    return name.lower()

df["original_name"] = df["source_image"].apply(original_name)

print("===== ẢNH GỐC VÀ BIẾN THỂ =====")

display(
    df.groupby("source_dataset").agg(
        records=("record_id", "count"),
        source_images=("source_image", "nunique"),
        original_names=("original_name", "nunique"),
        current_groups=("image_group", "nunique")
    ).reset_index()
)

print("\n===== TÊN FILE MẪU =====")

for source, part in df.groupby("source_dataset"):
    print("\nDATASET:", source)
    print(
        part["original_name"]
        .drop_duplicates()
        .head(12)
        .tolist()
    )

In [ ]:
print("===== ẢNH CÙNG TÊN NGUỒN =====")

source_groups = (
    df.groupby(["source_dataset", "original_name"])
    .agg(
        records=("record_id", "count"),
        classes=("target_class", "nunique"),
        source_files=("source_image", "nunique")
    )
    .reset_index()
)

print(
    "Số nhóm có nhiều hơn một bản ghi:",
    (source_groups["records"] > 1).sum()
)

print(
    "Nhóm nguồn chứa nhiều nhãn:",
    (source_groups["classes"] > 1).sum()
)

display(
    source_groups.sort_values(
        "records", ascending=False
    ).head(20)
)

source_groups.to_csv(
    REPORT_DIR / "v4_original_source_groups.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
pork = df[df["source_dataset"] == "pork"].copy()

display(
    pork.groupby("original_label").agg(
        crops=("record_id", "count"),
        source_images=("source_image", "nunique"),
        original_names=("original_name", "nunique")
    ).reset_index()
)

print("\n20 ảnh nguồn Pork mẫu:")

display(
    pork[
        [
            "original_label",
            "source_split",
            "original_name",
            "image_group"
        ]
    ].drop_duplicates().head(20)
)

In [ ]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
import pandas as pd
import re

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "manifest.csv",
    low_memory=False
).reset_index(drop=True)

required = [
    "source_dataset", "source_image", "target_class",
    "crop_path", "sha256"
]

assert all(c in df.columns for c in required), (
    "Manifest thiếu cột bắt buộc"
)

# Chuẩn hóa tên file Roboflow
def original_name(path):
    name = PurePosixPath(
        str(path).replace("\\", "/")
    ).name.lower()

    name = re.sub(
        r"\.rf\.[0-9a-f]+(?=\.[^.]+$)",
        "",
        name
    )
    return name

df["original_name"] = df["source_image"].apply(
    original_name
)

parent = list(range(len(df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Nhóm dựa trên nguồn gốc và ảnh trùng hoàn toàn
seen = {}

for i, row in enumerate(df.itertuples(index=False)):

    keys = [
        ("source", row.source_dataset, row.source_image),
        ("original", row.source_dataset, row.original_name),
        ("sha256", str(row.sha256).lower())
    ]

    for key in keys:
        if key in seen:
            union(i, seen[key])
        else:
            seen[key] = i

df["provisional_group"] = [
    find(i) for i in range(len(df))
]

print("Tổng records:", len(df))
print("Tổng nhóm sơ bộ:", df["provisional_group"].nunique())

display(
    df.groupby("source_dataset").agg(
        records=("crop_path", "count"),
        groups=("provisional_group", "nunique"),
        images=("source_image", "nunique")
    ).reset_index()
)

In [ ]:
group_stats = (
    df.groupby("provisional_group")
    .agg(
        records=("crop_path", "count"),
        classes=("target_class", "nunique"),
        sources=("source_image", "nunique")
    )
    .reset_index()
)

conflict_ids = set(
    group_stats.loc[
        group_stats["classes"] > 1,
        "provisional_group"
    ]
)

print("Tổng nhóm:", len(group_stats))
print("Nhóm chứa nhiều lớp:", len(conflict_ids))

conflicts = df[
    df["provisional_group"].isin(conflict_ids)
].copy()

display(
    conflicts[
        [
            "source_dataset", "original_name",
            "target_class", "provisional_group"
        ]
    ].drop_duplicates().head(40)
)

conflicts.to_csv(
    REPORT_DIR / "v4_provisional_group_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
df.to_csv(
    REPORT_DIR / "v4_provisional_group_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

group_stats.to_csv(
    REPORT_DIR / "v4_provisional_group_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu manifest nhóm sơ bộ.")

print("\n10 NHÓM LỚN NHẤT:")

display(
    group_stats.sort_values(
        "records",
        ascending=False
    ).head(10)
)

print("\nSỐ NHÓM THEO DATASET:")

display(
    df.groupby("source_dataset")[
        "provisional_group"
    ].nunique().reset_index(name="groups")
)

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import pandas as pd
import math

# ==================================================
# 1. ĐƯỜNG DẪN PROJECT
# ==================================================

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

REPORT_DIR = (
    PROJECT_DIR / "datasets" / "reports" / "state_v4"
)

MANIFEST_PATH = (
    REPORT_DIR / "v4_provisional_group_manifest.csv"
)

REVIEW_DIR = REPORT_DIR / "group_conflict_review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(MANIFEST_PATH, low_memory=False)

# Kiểm tra cột bắt buộc
required_cols = [
    "provisional_group",
    "target_class",
    "crop_path"
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Thiếu cột: {col}")


# ==================================================
# 2. TÌM NHÓM CÓ NHIỀU NHÃN
# ==================================================

group_stats = (
    df.groupby("provisional_group")
    .agg(
        records=("crop_path", "count"),
        classes=("target_class", "nunique")
    )
    .reset_index()
)

conflict_ids = group_stats.loc[
    group_stats["classes"] > 1,
    "provisional_group"
].tolist()

print("Tổng nhóm có nhiều nhãn:", len(conflict_ids))


# ==================================================
# 3. HÀM XỬ LÝ ĐƯỜNG DẪN ẢNH
# ==================================================

def resolve_path(path):
    path = Path(str(path))

    if path.is_absolute():
        return path

    return PROJECT_DIR / path


# ==================================================
# 4. TẠO CONTACT SHEET
# ==================================================

for gid in conflict_ids:

    part = df[
        df["provisional_group"] == gid
    ].copy()

    # Lấy tối đa 4 ảnh mỗi lớp
    # Không sử dụng include_groups=False
    sample_parts = []

    for class_name, class_df in part.groupby("target_class"):

        sampled = class_df.sample(
            n=min(4, len(class_df)),
            random_state=42
        )

        sample_parts.append(sampled)

    samples = pd.concat(
        sample_parts,
        ignore_index=True
    )

    tile_w = 240
    tile_h = 230

    cols = 4
    rows = math.ceil(len(samples) / cols)

    sheet = Image.new(
        "RGB",
        (cols * tile_w, rows * tile_h),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for i, item in samples.iterrows():

        image_path = resolve_path(item["crop_path"])

        if not image_path.exists():
            print("Không tìm thấy:", image_path)
            continue

        with Image.open(image_path) as im:

            thumb = ImageOps.contain(
                im.convert("RGB"),
                (220, 185)
            )

        x = (i % cols) * tile_w
        y = (i // cols) * tile_h

        sheet.paste(thumb, (x, y))

        # Giữ nguyên cột target_class
        class_name = str(item["target_class"])

        draw.text(
            (x + 5, y + 192),
            class_name[:32],
            fill="black"
        )

    output_path = REVIEW_DIR / f"group_{gid}.jpg"

    sheet.save(
        output_path,
        quality=92
    )

    print("\n" + "=" * 55)
    print("GROUP ID:", gid)
    print("Số bản ghi:", len(part))
    print("Số lớp:", part["target_class"].nunique())
    print("Nhãn:", part["target_class"].unique().tolist())

    display(sheet)


print("\nHOÀN THÀNH!")
print("Số contact sheet:", len(conflict_ids))
print("Thư mục lưu:", REVIEW_DIR)

In [ ]:
pairs = pd.read_csv(
    REPORT_DIR / "audit_pairs.csv",
    low_memory=False
)

def norm_path(p):
    return str(p).replace("\\", "/").casefold()

pairs["same_path"] = [
    norm_path(a) == norm_path(b)
    for a, b in zip(pairs["first"], pairs["second"])
]

pairs["cross_label"] = (
    pairs["first_class"] != pairs["second_class"]
)

print("===== THỐNG KÊ PHASH =====")

display(
    pairs.groupby(
        ["kind", "same_path", "cross_label"]
    ).size().reset_index(name="pairs")
)

print("\nCẶP KHÁC NHÃN:")

cross_label_pairs = pairs[
    ~pairs["same_path"] & pairs["cross_label"]
].copy()

display(cross_label_pairs.head(20))

print("Tổng cặp khác nhãn:", len(cross_label_pairs))

cross_label_pairs.to_csv(
    REPORT_DIR / "v4_cross_label_phash_review.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Mỗi đường dẫn crop ứng với một nhóm sơ bộ
path_to_group = {
    norm_path(path): gid
    for path, gid in zip(
        df["crop_path"],
        df["provisional_group"]
    )
}

pairs["first_group"] = pairs["first"].map(
    lambda p: path_to_group.get(norm_path(p))
)

pairs["second_group"] = pairs["second"].map(
    lambda p: path_to_group.get(norm_path(p))
)

known = pairs[
    pairs["first_group"].notna()
    & pairs["second_group"].notna()
].copy()

cross_group = known[
    known["first_group"] != known["second_group"]
]

print("Tổng cặp pHash:", len(pairs))
print("Cặp có đủ thông tin nhóm:", len(known))
print("Cặp nối hai nhóm khác nhau:", len(cross_group))

display(
    cross_group[
        [
            "kind", "distance",
            "first_class", "second_class",
            "first_group", "second_group"
        ]
    ].head(20)
)

cross_group.to_csv(
    REPORT_DIR / "v4_cross_group_phash_review.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

pairs = pd.read_csv(
    REPORT_DIR / "v4_cross_group_phash_review.csv",
    low_memory=False
)

print("Tổng cặp giữa các nhóm:", len(pairs))

print("\nPHÂN LOẠI THEO KIND:")
display(
    pairs.groupby("kind")
    .size()
    .reset_index(name="pairs")
)

print("\nPHÂN BỐ KHOẢNG CÁCH PHASH:")
display(
    pairs.groupby(["kind", "distance"])
    .size()
    .reset_index(name="pairs")
    .sort_values(["kind", "distance"])
)

print("\nPHÂN BỐ THEO LỚP:")
display(
    pairs.groupby("first_class")
    .size()
    .reset_index(name="pairs")
    .sort_values("pairs", ascending=False)
)

In [ ]:
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import math

REVIEW_DIR = REPORT_DIR / "cross_group_review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

def resolve_image_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_DIR / path

# Chọn tối đa 4 cặp cho mỗi tổ hợp kind/lớp
selected_parts = []

for (_, _), part in pairs.groupby(
    ["kind", "first_class"]
):
    selected_parts.append(
        part.sample(
            n=min(4, len(part)),
            random_state=42
        )
    )

selected = (
    pd.concat(selected_parts, ignore_index=True)
    .sort_values(["kind", "distance"])
    .head(24)
    .reset_index(drop=True)
)

tile_w, tile_h = 220, 200
pair_w = tile_w * 2
pair_h = tile_h + 45

columns = 2
rows = math.ceil(len(selected) / columns)

sheet = Image.new(
    "RGB",
    (columns * pair_w, rows * pair_h),
    "white"
)

draw = ImageDraw.Draw(sheet)

for i, item in selected.iterrows():
    col = i % columns
    row = i // columns

    x0 = col * pair_w
    y0 = row * pair_h

    for side, field in enumerate(["first", "second"]):
        path = resolve_image_path(item[field])

        if not path.exists():
            print("Thiếu ảnh:", path)
            continue

        with Image.open(path) as im:
            thumb = ImageOps.contain(
                im.convert("RGB"),
                (tile_w - 8, tile_h - 8)
            )

        x = x0 + side * tile_w
        sheet.paste(thumb, (x, y0))

    label = (
        f"{i} | {item['kind']} | "
        f"d={item['distance']} | "
        f"{str(item['first_class'])[:22]}"
    )

    draw.text(
        (x0 + 5, y0 + tile_h + 5),
        label,
        fill="black"
    )

output = REVIEW_DIR / "cross_group_sample.jpg"
sheet.save(output, quality=92)

selected.to_csv(
    REVIEW_DIR / "cross_group_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

display(sheet)

print("Đã lưu:", output)

In [ ]:
group_manifest = pd.read_csv(
    REPORT_DIR / "v4_provisional_group_manifest.csv",
    low_memory=False
)

# Các nhóm có cặp ảnh giống/gần giống
risky_groups = set(
    pairs["first_group"].dropna().astype(int)
) | set(
    pairs["second_group"].dropna().astype(int)
)

group_manifest["phash_review_required"] = (
    group_manifest["provisional_group"].isin(risky_groups)
)

print("Tổng nhóm sơ bộ:",
      group_manifest["provisional_group"].nunique())

print("Nhóm cần kiểm tra pHash:",
      len(risky_groups))

print("Số bản ghi thuộc nhóm cần kiểm tra:",
      group_manifest["phash_review_required"].sum())

group_manifest.to_csv(
    REPORT_DIR / "v4_group_review_flags.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "v4_provisional_group_manifest.csv",
    low_memory=False
)

pairs = pd.read_csv(
    REPORT_DIR / "audit_pairs.csv",
    low_memory=False
)

def normalize_path(p):
    return str(p).replace("\\", "/").casefold()

# Khởi tạo Union-Find theo các nhóm sơ bộ
group_ids = df["provisional_group"].astype(int).unique()
parent = {int(g): int(g) for g in group_ids}

def find(g):
    g = int(g)
    if parent[g] != g:
        parent[g] = find(parent[g])
    return parent[g]

def union(a, b):
    a, b = find(a), find(b)
    if a != b:
        parent[max(a, b)] = min(a, b)

path_to_group = {
    normalize_path(path): int(group)
    for path, group in zip(
        df["crop_path"],
        df["provisional_group"]
    )
}

# Gom các cặp exact_phash vào cùng nhóm
exact_pairs = 0

for row in pairs.itertuples(index=False):
    if row.kind not in ("exact_phash", "sha256"):
        continue

    a = path_to_group.get(normalize_path(row.first))
    b = path_to_group.get(normalize_path(row.second))

    if a is not None and b is not None:
        union(a, b)
        exact_pairs += 1

df["safe_group"] = [
    find(g) for g in df["provisional_group"]
]

# Những cặp near_phash chỉ đánh dấu kiểm duyệt
risky_groups = set()

for row in pairs.itertuples(index=False):
    if row.kind != "near_phash":
        continue

    a = path_to_group.get(normalize_path(row.first))
    b = path_to_group.get(normalize_path(row.second))

    if a is not None:
        risky_groups.add(find(a))
    if b is not None:
        risky_groups.add(find(b))

df["near_phash_review"] = (
    df["safe_group"].isin(risky_groups)
)

# Không tự coi nhóm sơ bộ là phiên thu thập độc lập
df["split_eligibility"] = "source_review_required"

# Lưu kết quả
output = REPORT_DIR / "v4_safe_group_manifest.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("===== KẾT QUẢ =====")
print("Nhóm ban đầu:", len(group_ids))
print("Cặp exact được xét:", exact_pairs)
print("Nhóm sau khi gom:", df["safe_group"].nunique())
print("Nhóm liên quan near_phash:", len(risky_groups))
print("Bản ghi cần xem near_phash:",
      int(df["near_phash_review"].sum()))
print("Đã lưu:", output)

In [ ]:
group_check = (
    df.groupby("safe_group")
    .agg(
        records=("crop_path", "count"),
        classes=("target_class", "nunique"),
        sources=("source_image", "nunique"),
        near_review=("near_phash_review", "any")
    )
    .reset_index()
)

print("Tổng nhóm:", len(group_check))
print(
    "Nhóm có nhiều nhãn:",
    int((group_check["classes"] > 1).sum())
)
print(
    "Nhóm cần kiểm duyệt near_phash:",
    int(group_check["near_review"].sum())
)

print("\n10 NHÓM LỚN NHẤT:")
display(
    group_check.sort_values(
        "records", ascending=False
    ).head(10)
)

print("\nSỐ NHÓM THEO NGUYÊN LIỆU:")
display(
    df.groupby("source_dataset")
    .agg(
        records=("crop_path", "count"),
        groups=("safe_group", "nunique")
    )
    .reset_index()
)

group_check.to_csv(
    REPORT_DIR / "v4_safe_group_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "v4_safe_group_manifest.csv",
    low_memory=False
)

pairs = pd.read_csv(
    REPORT_DIR / "audit_pairs.csv",
    low_memory=False
)

def norm_path(value):
    return str(value).replace("\\", "/").casefold()

path_group = {
    norm_path(path): int(group)
    for path, group in zip(
        df["crop_path"],
        df["safe_group"]
    )
}

pairs["group_a"] = pairs["first"].map(
    lambda p: path_group.get(norm_path(p))
)

pairs["group_b"] = pairs["second"].map(
    lambda p: path_group.get(norm_path(p))
)

near = pairs[
    (pairs["kind"] == "near_phash")
    & pairs["group_a"].notna()
    & pairs["group_b"].notna()
    & (pairs["group_a"] != pairs["group_b"])
].copy()

near["distance"] = pd.to_numeric(
    near["distance"], errors="coerce"
)

print("Cặp near_phash nối hai nhóm khác nhau:", len(near))

display(
    near.groupby("distance")
    .size()
    .reset_index(name="pairs")
    .sort_values("distance")
)

near.to_csv(
    REPORT_DIR / "v4_near_pairs_pending.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import math

REVIEW_DIR = REPORT_DIR / "near_phash_review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

def resolve_image(path):
    p = Path(str(path))
    return p if p.is_absolute() else PROJECT_DIR / p

selected_parts = []

for (_, _), part in near.groupby(
    ["distance", "first_class"],
    dropna=False
):
    selected_parts.append(
        part.sample(
            n=min(3, len(part)),
            random_state=42
        )
    )

if not selected_parts:
    print("Không có cặp near_phash cần kiểm tra.")
else:
    selected = (
        pd.concat(selected_parts, ignore_index=True)
        .sort_values(["distance", "first_class"])
        .reset_index(drop=True)
    )

    # Giới hạn 48 cặp cho lượt kiểm tra đầu
    selected = selected.head(48).copy()
    selected["decision"] = "pending"

    tile_w = 200
    tile_h = 175
    pair_w = 400
    pair_h = 220
    columns = 2
    rows = math.ceil(len(selected) / columns)

    sheet = Image.new(
        "RGB",
        (pair_w * columns, pair_h * rows),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for i, item in selected.iterrows():
        x0 = (i % columns) * pair_w
        y0 = (i // columns) * pair_h

        for side, field in enumerate(["first", "second"]):
            image_path = resolve_image(item[field])

            if not image_path.exists():
                print("Thiếu ảnh:", image_path)
                continue

            with Image.open(image_path) as im:
                thumb = ImageOps.contain(
                    im.convert("RGB"),
                    (tile_w - 8, tile_h - 8)
                )

            sheet.paste(thumb, (x0 + side * tile_w, y0))

        label = (
            f"Pair {i} | d={item['distance']} | "
            f"{str(item['first_class'])[:20]}"
        )

        draw.text(
            (x0 + 5, y0 + 183),
            label,
            fill="black"
        )

    sheet_path = REVIEW_DIR / "near_pairs_sample.jpg"
    sheet.save(sheet_path, quality=90)

    selected.to_csv(
        REVIEW_DIR / "near_pairs_sample.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(sheet)
    print("Đã lưu:", sheet_path)

In [ ]:
# Những nhóm đang có rủi ro near_phash
risky_groups = (
    set(near["group_a"].astype(int))
    | set(near["group_b"].astype(int))
)

group_summary = (
    df.groupby("safe_group")
    .agg(
        records=("crop_path", "count"),
        classes=("target_class", "nunique"),
        sources=("source_image", "nunique")
    )
    .reset_index()
)

group_summary["near_phash_risk"] = (
    group_summary["safe_group"].isin(risky_groups)
)

group_summary["split_status"] = (
    "source_review_required"
)

group_summary["review_priority"] = (
    group_summary["near_phash_risk"].map({
        True: "high",
        False: "normal"
    })
)

output = REPORT_DIR / "v4_split_readiness.csv"

group_summary.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("Tổng nhóm:", len(group_summary))
print(
    "Nhóm có liên kết near_phash qua nhóm khác:",
    group_summary["near_phash_risk"].sum()
)
print(
    "Nhóm không có liên kết near_phash được báo cáo:",
    (~group_summary["near_phash_risk"]).sum()
)

print("\nĐã lưu:", output)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "v4_safe_group_manifest.csv",
    low_memory=False
)

near = pd.read_csv(
    REPORT_DIR / "v4_near_pairs_pending.csv",
    low_memory=False
)

group_ids = df["safe_group"].dropna().astype(int).unique()
parent = {int(g): int(g) for g in group_ids}

def find(x):
    x = int(x)

    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]

    return x

def union(a, b):
    ra, rb = find(a), find(b)

    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

# Gộp liên kết gần trùng để tránh chia chúng sang hai split
for row in near.itertuples(index=False):
    if pd.notna(row.group_a) and pd.notna(row.group_b):
        union(int(row.group_a), int(row.group_b))

df["leakage_group"] = [
    find(g) for g in df["safe_group"]
]

print("Nhóm trước:", df["safe_group"].nunique())
print("Nhóm sau:", df["leakage_group"].nunique())

df.to_csv(
    REPORT_DIR / "v4_leakage_group_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
group_summary = (
    df.groupby("leakage_group")
    .agg(
        records=("crop_path", "count"),
        classes=("target_class", "nunique"),
        source_images=("source_image", "nunique"),
        datasets=("source_dataset", "nunique")
    )
    .reset_index()
)

print("Tổng nhóm:", len(group_summary))
print("Nhóm lớn nhất:", group_summary["records"].max())
print("Nhóm chứa nhiều lớp:",
      (group_summary["classes"] > 1).sum())

print("\n10 NHÓM LỚN NHẤT:")
display(
    group_summary.sort_values(
        "records", ascending=False
    ).head(10)
)

print("\nSỐ NHÓM THEO DATASET:")
display(
    df.groupby("source_dataset")["leakage_group"]
    .nunique()
    .reset_index(name="groups")
)

group_summary.to_csv(
    REPORT_DIR / "v4_leakage_group_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "v4_leakage_group_manifest.csv",
    low_memory=False
)

mixed = (
    df.groupby("leakage_group")
    .filter(lambda x: x["target_class"].nunique() > 1)
)

review = (
    mixed.groupby(
        ["leakage_group", "target_class"]
    )
    .agg(
        records=("crop_path", "count"),
        source_images=("source_image", "nunique")
    )
    .reset_index()
)

display(review)

review.to_csv(
    REPORT_DIR / "v4_mixed_groups_final_review.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import PurePosixPath
import re

def get_stem(path):
    name = PurePosixPath(
        str(path).replace("\\", "/")
    ).name.lower()

    return re.sub(
        r"\.rf\.[0-9a-f]+(?=\.[^.]+$)",
        "",
        name
    )

df["original_name"] = df["source_image"].apply(get_stem)

print("ẢNH NGUỒN THEO DATASET:")

display(
    df.groupby("source_dataset").agg(
        records=("crop_path", "count"),
        original_names=("original_name", "nunique"),
        leakage_groups=("leakage_group", "nunique")
    ).reset_index()
)

for source in ["lettuce", "pork", "onion"]:
    part = df[df["source_dataset"] == source]

    print(f"\n===== {source.upper()} =====")
    print("\n".join(part["original_name"].drop_duplicates().head(15)))

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "v4_leakage_group_manifest.csv",
    low_memory=False
)

def extract_capture_time(row):
    name = Path(
        str(row["source_image"]).replace("\\", "/")
    ).name.lower()

    name = re.sub(
        r"\.rf\.[a-f0-9]+(?=\.[^.]+$)", "", name
    )

    if row["source_dataset"] == "lettuce":
        # win_20260429_17_15_51_pro_jpg.jpg
        m = re.search(
            r"win_(\d{8})_(\d{2})_(\d{2})_(\d{2})",
            name
        )
        if m:
            return pd.to_datetime(
                "".join(m.groups()),
                format="%Y%m%d%H%M%S",
                errors="coerce"
            )

    if row["source_dataset"] == "pork":
        # 1758607361776-1-_jpg.jpg
        m = re.match(r"(\d{13})(?:[-_]|$)", name)

        if m:
            return pd.to_datetime(
                int(m.group(1)),
                unit="ms",
                utc=True,
                errors="coerce"
            ).tz_convert(None)

    return pd.NaT

df["capture_time"] = df.apply(
    extract_capture_time, axis=1
)

print("===== ẢNH CÓ DẤU THỜI GIAN =====")

display(
    df.groupby("source_dataset").agg(
        records=("crop_path", "count"),
        timestamps=("capture_time", "count"),
        first_time=("capture_time", "min"),
        last_time=("capture_time", "max")
    ).reset_index()
)

In [ ]:
records = []

for source in ["lettuce", "pork"]:

    part = (
        df[
            (df["source_dataset"] == source)
            & df["capture_time"].notna()
        ]
        .drop_duplicates("source_image")
        .sort_values("capture_time")
        .copy()
    )

    if part.empty:
        continue

    gaps = part["capture_time"].diff()

    part["candidate_session"] = (
        gaps.isna()
        | (gaps > pd.Timedelta(minutes=30))
    ).cumsum()

    part["session_key"] = (
        source + "_"
        + part["candidate_session"].astype(str)
    )

    records.append(part)

if records:
    sessions_df = pd.concat(
        records, ignore_index=True
    )

    summary = (
        sessions_df.groupby(
            ["source_dataset", "session_key"]
        )
        .agg(
            images=("source_image", "nunique"),
            start=("capture_time", "min"),
            end=("capture_time", "max"),
            classes=("target_class", "nunique")
        )
        .reset_index()
    )

    print("===== CỤM THỜI GIAN ỨNG VIÊN =====")
    display(summary)

    sessions_df.to_csv(
        REPORT_DIR / "v4_capture_session_candidates.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        REPORT_DIR / "v4_capture_session_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )
else:
    print("Không tìm thấy timestamp phù hợp.")

In [ ]:
print("===== SPLIT NGUỒN =====")

display(
    pd.crosstab(
        df["source_dataset"],
        df["source_split"]
    )
)

split_overlap = (
    df.groupby("leakage_group")["source_split"]
    .nunique()
)

print(
    "Số nhóm leakage xuất hiện trong nhiều split nguồn:",
    int((split_overlap > 1).sum())
)

print(
    "Tổng nhóm leakage:",
    df["leakage_group"].nunique()
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

df = pd.read_csv(
    REPORT_DIR / "v4_leakage_group_manifest.csv",
    low_memory=False
)

# Các nhóm xuất hiện trong nhiều split của Roboflow
split_counts = (
    df.groupby("leakage_group")["source_split"]
    .nunique()
)

overlap_ids = set(
    split_counts[split_counts > 1].index
)

df["split_overlap"] = df["leakage_group"].isin(overlap_ids)

print("Số nhóm chồng chéo:", len(overlap_ids))
print("Bản ghi liên quan:", int(df["split_overlap"].sum()))

print("\n===== THỐNG KÊ THEO DATASET =====")

display(
    df.groupby("source_dataset")
    .agg(
        records=("crop_path", "count"),
        overlap_records=("split_overlap", "sum"),
        total_groups=("leakage_group", "nunique"),
        overlap_groups=(
            "leakage_group",
            lambda s: s[s.isin(overlap_ids)].nunique()
        )
    )
    .reset_index()
)

print("\n===== CÁC SPLIT CHỒNG CHÉO =====")

overlap_report = (
    df[df["split_overlap"]]
    .groupby(["source_dataset", "leakage_group"])
    .agg(
        records=("crop_path", "count"),
        splits=(
            "source_split",
            lambda s: ",".join(sorted(s.dropna().unique()))
        ),
        classes=("target_class", "nunique")
    )
    .reset_index()
)

display(overlap_report.head(20))

overlap_report.to_csv(
    REPORT_DIR / "v4_split_overlap_diagnostic.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import numpy as np

# Những nhóm chứa nhiều nhãn
mixed_groups = set(
    df.groupby("leakage_group")["target_class"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)

df["mixed_group"] = df["leakage_group"].isin(mixed_groups)

df["proposed_usage"] = np.where(
    df["split_overlap"] | df["mixed_group"],
    "review_required",
    "train_candidate"
)

# Chỉ là kế hoạch sơ bộ, chưa xác nhận nguồn độc lập
df["approved_for_training"] = False

output = REPORT_DIR / "v4_proposed_usage.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("===== KẾ HOẠCH SỬ DỤNG V4 =====")

display(
    df.groupby(
        ["source_dataset", "proposed_usage"]
    )
    .agg(
        records=("crop_path", "count"),
        groups=("leakage_group", "nunique")
    )
    .reset_index()
)

print("\nĐã lưu:", output)

In [ ]:
from pathlib import Path
from collections import Counter

V3_DIR = PROJECT_DIR / "datasets" / "efficientnet_v3"

EXTS = {".jpg", ".jpeg", ".png", ".webp"}

records = []

for split in ["train", "val", "test"]:
    split_dir = V3_DIR / split

    if not split_dir.exists():
        continue

    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue

        images = [
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in EXTS
        ]

        records.append({
            "split": split,
            "class": class_dir.name,
            "images": len(images)
        })

v3_summary = pd.DataFrame(records)

print("Tổng ảnh V3:", v3_summary["images"].sum())
print("Số lớp V3:", v3_summary["class"].nunique())

display(
    v3_summary.pivot_table(
        index="class",
        columns="split",
        values="images",
        aggfunc="sum",
        fill_value=0
    )
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATA_DIR = PROJECT_DIR / "datasets"
REPORT_DIR = DATA_DIR / "reports" / "state_v4"

EXTS = {".jpg", ".jpeg", ".png", ".webp"}

records = []

# V3: giữ thông tin split hiện tại
v3_dir = DATA_DIR / "efficientnet_v3"

for split in ["train", "val", "test"]:
    for class_dir in sorted((v3_dir / split).iterdir()):
        if not class_dir.is_dir():
            continue

        for path in class_dir.iterdir():
            if path.is_file() and path.suffix.lower() in EXTS:
                records.append({
                    "dataset": "v3",
                    "path": str(path.resolve()),
                    "target_class": class_dir.name,
                    "original_split": split,
                    "usage": "existing_split",
                    "group_id": ""
                })

# V4: giữ trạng thái ứng viên, không dùng split Roboflow làm split cuối
v4 = pd.read_csv(
    REPORT_DIR / "v4_proposed_usage.csv",
    low_memory=False
)

for row in v4.itertuples(index=False):
    p = Path(str(row.crop_path))
    if not p.is_absolute():
        p = PROJECT_DIR / p

    records.append({
        "dataset": "v4",
        "path": str(p.resolve()),
        "target_class": row.target_class,
        "original_split": "",
        "usage": row.proposed_usage,
        "group_id": f"v4:{row.leakage_group}"
    })

# Bắp cải: dữ liệu ứng viên từ video
cabbage_dir = (
    DATA_DIR / "efficientnet_v4_cabbage_candidates" / "train"
)

if cabbage_dir.exists():
    for class_dir in cabbage_dir.iterdir():
        if not class_dir.is_dir():
            continue

        for p in class_dir.iterdir():
            if p.is_file() and p.suffix.lower() in EXTS:
                state_group = (
                    "fresh_video"
                    if class_dir.name.endswith("normal_appearance")
                    else "rotten_video"
                )

                records.append({
                    "dataset": "cabbage",
                    "path": str(p.resolve()),
                    "target_class": class_dir.name,
                    "original_split": "",
                    "usage": "train_candidate",
                    "group_id": f"cabbage:{state_group}"
                })

combined = pd.DataFrame(records)

# Không tính nhiều provenance records trỏ đến một file thành nhiều ảnh
combined = combined.drop_duplicates(
    subset=["dataset", "path", "target_class"]
).reset_index(drop=True)

print("===== TỔNG HỢP =====")
display(
    combined.groupby("dataset").agg(
        images=("path", "nunique"),
        classes=("target_class", "nunique")
    )
)

print("\nTổng file tham chiếu:", len(combined))
print("Tổng lớp có dữ liệu:", combined["target_class"].nunique())

print(
    "Đường dẫn thiếu:",
    (~combined["path"].map(lambda x: Path(x).is_file())).sum()
)

combined.to_csv(
    REPORT_DIR / "efficientnet_combined_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import hashlib
from tqdm.auto import tqdm

def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

if not combined["path"].map(lambda p: Path(p).is_file()).all():
    raise RuntimeError("Còn đường dẫn ảnh thiếu.")

# Hash mỗi đường dẫn một lần
unique_paths = combined["path"].unique()

hash_map = {
    p: file_sha256(p)
    for p in tqdm(unique_paths, desc="SHA256 toàn bộ dataset")
}

combined["sha256"] = combined["path"].map(hash_map)

# Hash xuất hiện trong nhiều lớp
label_conflicts = (
    combined.groupby("sha256")["target_class"].nunique()
)

# Hash xuất hiện ở các split V3 khác nhau
v3 = combined[combined["dataset"] == "v3"].copy()

split_conflicts = (
    v3.groupby("sha256")["original_split"].nunique()
)

# Hash xuất hiện trong nhiều dataset
source_overlap = (
    combined.groupby("sha256")["dataset"].nunique()
)

print("Ảnh SHA256 duy nhất:", combined["sha256"].nunique())
print("Nhóm SHA256 có nhiều lớp:",
      int((label_conflicts > 1).sum()))
print("Nhóm SHA256 trùng giữa split V3:",
      int((split_conflicts > 1).sum()))
print("Nhóm SHA256 xuất hiện ở nhiều dataset:",
      int((source_overlap > 1).sum()))

combined.to_csv(
    REPORT_DIR / "efficientnet_combined_hashed.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import pandas as pd
import imagehash

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

INPUT_PATH = REPORT_DIR / "efficientnet_combined_hashed.csv"
OUTPUT_PATH = REPORT_DIR / "efficientnet_combined_phash.csv"

combined = pd.read_csv(INPUT_PATH, low_memory=False)

# Hỗ trợ chạy tiếp nếu đã có kết quả pHash cũ
if OUTPUT_PATH.exists():
    old = pd.read_csv(
        OUTPUT_PATH,
        usecols=["path", "phash"],
        dtype=str
    )
    cache = dict(zip(old["path"], old["phash"]))
else:
    cache = {}

def calculate_phash(path):
    with Image.open(path) as im:
        image = ImageOps.exif_transpose(im).convert("RGB")
        return str(imagehash.phash(image))

missing = [
    p for p in combined["path"].unique()
    if p not in cache or pd.isna(cache[p])
]

print("Tổng ảnh:", len(combined))
print("Cần tính pHash:", len(missing))

errors = []

for path in tqdm(missing, desc="Computing pHash"):
    try:
        cache[path] = calculate_phash(path)
    except Exception as e:
        errors.append((path, str(e)))

combined["phash"] = combined["path"].map(cache)

combined.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Đã tính:", combined["phash"].notna().sum())
print("Ảnh lỗi:", len(errors))
print("Đã lưu:", OUTPUT_PATH)

if errors:
    display(pd.DataFrame(errors, columns=["path", "error"]).head())

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

df = pd.read_csv(
    REPORT_DIR / "efficientnet_combined_phash.csv",
    low_memory=False
)

if df["phash"].isna().any():
    raise RuntimeError("Còn ảnh thiếu pHash.")

df["hash_int"] = df["phash"].apply(
    lambda h: int(str(h), 16)
)

# Tập V3 Validation/Test cần bảo vệ
protected = df[
    (df["dataset"] == "v3")
    & df["original_split"].isin(["val", "test"])
].copy().reset_index(drop=True)

# Tập cần so sánh: V3 Train + V4 + Bắp cải
candidates = df[
    ~(
        (df["dataset"] == "v3")
        & df["original_split"].isin(["val", "test"])
    )
].copy().reset_index(drop=True)

print("Ảnh V3 Val/Test:", len(protected))
print("Ảnh cần đối chiếu:", len(candidates))

# Chia 64 bit thành 6 khối: 11, 11, 11, 11, 10, 10
sizes = [11, 11, 11, 11, 10, 10]
offsets = []

offset = 0
for size in sizes:
    offsets.append(offset)
    offset += size

def blocks(h):
    for i, (start, size) in enumerate(zip(offsets, sizes)):
        mask = (1 << size) - 1
        yield (i, (h >> start) & mask)

index = defaultdict(list)

for i, h in enumerate(protected["hash_int"]):
    for key in blocks(int(h)):
        index[key].append(i)

matches = []

for row in tqdm(
    candidates.itertuples(index=False),
    total=len(candidates),
    desc="Checking V3 Val/Test leakage"
):
    h = int(row.hash_int)

    possible = set()

    for key in blocks(h):
        possible.update(index.get(key, []))

    for j in possible:
        ref = protected.iloc[j]
        distance = (h ^ int(ref["hash_int"])).bit_count()

        if distance <= 5:
            matches.append({
                "candidate_path": row.path,
                "candidate_dataset": row.dataset,
                "candidate_class": row.target_class,
                "protected_path": ref["path"],
                "protected_split": ref["original_split"],
                "protected_class": ref["target_class"],
                "phash_distance": distance,
                "same_class": (
                    row.target_class == ref["target_class"]
                )
            })

leakage_df = pd.DataFrame(
    matches,
    columns=[
        "candidate_path", "candidate_dataset",
        "candidate_class", "protected_path",
        "protected_split", "protected_class",
        "phash_distance", "same_class"
    ]
)

output = REPORT_DIR / "efficientnet_v3_protected_phash_pairs.csv"

leakage_df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== KẾT QUẢ =====")
print("Tổng cặp nghi gần trùng:", len(leakage_df))
print(
    "Số ảnh ứng viên bị ảnh hưởng:",
    leakage_df["candidate_path"].nunique()
)
print(
    "Cặp khác lớp:",
    int((~leakage_df["same_class"]).sum())
)

display(
    leakage_df.groupby(
        ["candidate_dataset", "protected_split"],
        dropna=False
    ).size().reset_index(name="pairs")
)

print("Đã lưu:", output)

In [ ]:
new_data = df[
    df["dataset"].isin(["v4", "cabbage"])
].copy()

suspect_paths = set(
    leakage_df["candidate_path"]
)

new_data["phash_review"] = (
    new_data["path"].isin(suspect_paths)
)

new_data["proposed_usage"] = "candidate_only"

new_data.loc[
    new_data["phash_review"],
    "proposed_usage"
] = "quarantine_pending_review"

print("===== KIỂM TRA V4 + BẮP CẢI =====")

display(
    new_data.groupby(
        ["dataset", "proposed_usage"]
    ).size().reset_index(name="images")
)

output = REPORT_DIR / "efficientnet_new_data_review.csv"

new_data.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", output)

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

pairs = pd.read_csv(
    REPORT_DIR / "efficientnet_v3_protected_phash_pairs.csv"
)

tile_w = 290
tile_h = 250
info_h = 85

sheet = Image.new(
    "RGB",
    (tile_w * 2, (tile_h + info_h) * len(pairs)),
    "white"
)

draw = ImageDraw.Draw(sheet)

for i, row in pairs.iterrows():

    y = i * (tile_h + info_h)

    for side, field in enumerate([
        "candidate_path", "protected_path"
    ]):
        p = Path(row[field])

        with Image.open(p) as im:
            thumb = ImageOps.contain(
                ImageOps.exif_transpose(im).convert("RGB"),
                (tile_w - 10, tile_h - 10)
            )

        sheet.paste(thumb, (side * tile_w, y))

    draw.text(
        (5, y + tile_h + 4),
        f"Pair {i} | pHash distance={row['phash_distance']}",
        fill="black"
    )

    draw.text(
        (5, y + tile_h + 25),
        f"TRAIN: {row['candidate_class']}",
        fill="black"
    )

    draw.text(
        (5, y + tile_h + 46),
        f"{row['protected_split'].upper()}: "
        f"{row['protected_class']}",
        fill="black"
    )

output = REPORT_DIR / "v3_phash_conflict_review.jpg"
sheet.save(output, quality=95)

display(sheet)

print("Đã lưu:", output)

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

df = pd.read_csv(
    REPORT_DIR / "efficientnet_combined_phash.csv",
    low_memory=False
)

v3 = df[df["dataset"] == "v3"].copy()

v3["hash_int"] = v3["phash"].apply(
    lambda x: int(str(x), 16)
)

val = v3[v3["original_split"] == "val"].reset_index(drop=True)
test = v3[v3["original_split"] == "test"].reset_index(drop=True)

# 64 bit chia thành 6 khối.
# Với khoảng cách Hamming <= 5,
# ít nhất một khối phải giống nhau.
sizes = [11, 11, 11, 11, 10, 10]

offsets = []
offset = 0

for size in sizes:
    offsets.append(offset)
    offset += size

def hash_blocks(h):
    for i, (start, size) in enumerate(zip(offsets, sizes)):
        yield (
            i,
            (int(h) >> start) & ((1 << size) - 1)
        )

index = defaultdict(list)

for i, h in enumerate(val["hash_int"]):
    for key in hash_blocks(h):
        index[key].append(i)

matches = []

for row in tqdm(
    test.itertuples(index=False),
    total=len(test),
    desc="Checking V3 Val vs Test"
):
    possible = set()

    for key in hash_blocks(row.hash_int):
        possible.update(index.get(key, []))

    for j in possible:
        ref = val.iloc[j]

        distance = (
            int(row.hash_int) ^ int(ref["hash_int"])
        ).bit_count()

        if distance <= 5:
            matches.append({
                "test_path": row.path,
                "test_class": row.target_class,
                "val_path": ref["path"],
                "val_class": ref["target_class"],
                "distance": distance,
                "same_class": (
                    row.target_class == ref["target_class"]
                )
            })

val_test_pairs = pd.DataFrame(
    matches,
    columns=[
        "test_path", "test_class",
        "val_path", "val_class",
        "distance", "same_class"
    ]
)

print("Ảnh Validation:", len(val))
print("Ảnh Test:", len(test))

print("Cặp gần trùng Val-Test:", len(val_test_pairs))

if len(val_test_pairs):
    print(
        "Cặp khác nhãn:",
        int((~val_test_pairs["same_class"]).sum())
    )
    display(val_test_pairs.head(20))

val_test_pairs.to_csv(
    REPORT_DIR / "v3_val_test_phash_review.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"

combined = pd.read_csv(
    REPORT_DIR / "efficientnet_combined_phash.csv",
    low_memory=False
)

v4 = pd.read_csv(
    REPORT_DIR / "v4_proposed_usage.csv",
    low_memory=False
)

def norm_path(p):
    p = Path(str(p))
    if not p.is_absolute():
        p = PROJECT_DIR / p
    return str(p.resolve()).replace("\\", "/").casefold()

# Tra cứu trạng thái V4 theo đường dẫn
v4["join_path"] = v4["crop_path"].map(norm_path)

# Một file có thể có nhiều bản ghi provenance.
# Nếu có bất kỳ bản ghi nào cần review thì ưu tiên review.
v4_status = (
    v4.groupby("join_path")["proposed_usage"]
    .apply(
        lambda s: "review_required"
        if (s == "review_required").any()
        else "train_candidate"
    )
    .to_dict()
)

plan = combined[
    [
        "dataset", "path", "target_class",
        "original_split", "sha256", "phash"
    ]
].copy()

plan["usage_plan"] = "pending_review"

# V3
plan.loc[
    (plan["dataset"] == "v3")
    & (plan["original_split"] == "train"),
    "usage_plan"
] = "v3_train_existing"

plan.loc[
    (plan["dataset"] == "v3")
    & (plan["original_split"] == "val"),
    "usage_plan"
] = "v3_validation_protected"

plan.loc[
    (plan["dataset"] == "v3")
    & (plan["original_split"] == "test"),
    "usage_plan"
] = "v3_test_protected"

# V4
v4_mask = plan["dataset"] == "v4"

plan.loc[v4_mask, "usage_plan"] = (
    plan.loc[v4_mask, "path"]
    .map(norm_path)
    .map(v4_status)
    .fillna("review_required")
)

# Bắp cải
plan.loc[
    plan["dataset"] == "cabbage",
    "usage_plan"
] = "cabbage_train_candidate"

# Chưa có bước phê duyệt nhãn cuối
plan["approved_for_training"] = False

output = REPORT_DIR / "efficientnet_training_plan_37.csv"

plan.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("===== KẾ HOẠCH DỮ LIỆU =====")

display(
    plan.groupby(
        ["dataset", "usage_plan"]
    ).agg(
        images=("path", "nunique"),
        classes=("target_class", "nunique")
    ).reset_index()
)

print("\nTổng file:", len(plan))
print("Tổng lớp:", plan["target_class"].nunique())
print("Đã lưu:", output)

In [ ]:
class_report = (
    plan.groupby(
        ["target_class", "usage_plan"]
    )
    .size()
    .unstack(fill_value=0)
)

class_report["total"] = class_report.sum(axis=1)

class_report = class_report.sort_values(
    "total", ascending=True
)

display(class_report)

class_report.to_csv(
    REPORT_DIR / "efficientnet_37_class_readiness.csv",
    encoding="utf-8-sig"
)

print("Số lớp có dữ liệu:", len(class_report))

print("\n15 LỚP ÍT ẢNH NHẤT:")
display(class_report.head(15))

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports" / "state_v4"
PILOT_DIR = PROJECT_DIR / "datasets" / "efficientnet_pilot_37"

PILOT_DIR.mkdir(parents=True, exist_ok=True)

plan = pd.read_csv(
    REPORT_DIR / "efficientnet_training_plan_37.csv",
    low_memory=False
)

# Chỉ chọn ảnh để chuẩn bị Train thử nghiệm
allowed_usage = [
    "v3_train_existing",
    "train_candidate",
    "cabbage_train_candidate"
]

pilot = plan[
    plan["usage_plan"].isin(allowed_usage)
].copy()

pilot = pilot.drop_duplicates(
    subset=["path"]
).reset_index(drop=True)

pilot["split"] = "train"

# Bảo vệ V3 Validation/Test
eval_data = plan[
    plan["usage_plan"].isin([
        "v3_validation_protected",
        "v3_test_protected"
    ])
].copy()

eval_data["split"] = eval_data["usage_plan"].map({
    "v3_validation_protected": "val",
    "v3_test_protected": "test"
})

# Kiểm tra ảnh tồn tại
missing = pilot[
    ~pilot["path"].map(lambda p: Path(str(p)).is_file())
]

if not missing.empty:
    raise RuntimeError(
        f"Có {len(missing)} đường dẫn ảnh không tồn tại!"
    )

# Lưu danh sách lớp
class_names = sorted(pilot["target_class"].unique())

with open(
    PILOT_DIR / "class_names.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        class_names,
        f,
        ensure_ascii=False,
        indent=2
    )

pilot.to_csv(
    PILOT_DIR / "train_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

eval_data.to_csv(
    PILOT_DIR / "v3_eval_reference.csv",
    index=False,
    encoding="utf-8-sig"
)

print("===== PILOT DATASET =====")
print("Ảnh Train ứng viên:", len(pilot))
print("Số lớp:", len(class_names))

print("V3 Validation:", (eval_data["split"] == "val").sum())
print("V3 Test:", (eval_data["split"] == "test").sum())

print("\nẢNH THEO NGUỒN:")
display(
    pilot.groupby("dataset")
    .size()
    .reset_index(name="images")
)

print("\nĐã lưu:", PILOT_DIR)

In [ ]:
class_counts = (
    pilot.groupby("target_class")
    .size()
    .sort_values()
    .reset_index(name="train_images")
)

print("===== PHÂN BỐ TRAIN 37 LỚP =====")
display(class_counts)

print("\nTHỐNG KÊ:")
print("Tổng lớp:", len(class_counts))
print("Lớp ít ảnh nhất:", class_counts.iloc[0].to_dict())
print("Lớp nhiều ảnh nhất:", class_counts.iloc[-1].to_dict())

print(
    "Lớp có dưới 50 ảnh:",
    int((class_counts["train_images"] < 50).sum())
)

class_counts.to_csv(
    PILOT_DIR / "class_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import sys
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        )
    )
else:
    print("Chưa phát hiện GPU CUDA, sẽ dùng CPU.")

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd
import torch
import json

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
PILOT_DIR = PROJECT_DIR / "datasets" / "efficientnet_pilot_37"

train_df = pd.read_csv(
    PILOT_DIR / "train_manifest.csv",
    low_memory=False
)

with open(
    PILOT_DIR / "class_names.json",
    "r",
    encoding="utf-8"
) as f:
    class_names = json.load(f)

class_to_idx = {
    name: i for i, name in enumerate(class_names)
}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class FoodStateDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        with Image.open(row["path"]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
            image = self.transform(image)

        label = class_to_idx[row["target_class"]]

        return image, label

dataset = FoodStateDataset(train_df, transform)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0  # Phù hợp Jupyter trên Windows
)

images, labels = next(iter(loader))

print("===== DATALOADER TEST =====")
print("Tổng ảnh:", len(dataset))
print("Tổng lớp:", len(class_names))
print("Batch images:", images.shape)
print("Batch labels:", labels.shape)
print("Nhãn batch:", labels.tolist())

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    len(class_names)
)

model = model.to(device)
model.eval()

with torch.no_grad():
    outputs = model(images.to(device))

print("===== MODEL TEST =====")
print("Thiết bị:", device)
print("Số lớp:", len(class_names))
print("Đầu ra:", outputs.shape)

assert outputs.shape == (8, 37)

print("PASS: DataLoader và EfficientNet-B0 tương thích.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm

# CPU
device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))

# Model được tạo ở Bước 157.3
model = model.to(device)

# Đóng băng backbone
for param in model.features.parameters():
    param.requires_grad = False

# Chỉ học classifier
for param in model.classifier.parameters():
    param.requires_grad = True

model.features.eval()
model.classifier.train()

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

MAX_BATCHES = 3

print("===== SMOKE TEST TRAINING =====")
print("Device:", device)
print("Classes:", len(class_names))
print("Batches:", MAX_BATCHES)

loss_history = []

for batch_idx, (images, labels) in enumerate(loader):

    if batch_idx >= MAX_BATCHES:
        break

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    loss_history.append(float(loss.item()))

    print(
        f"Batch {batch_idx + 1}/{MAX_BATCHES}"
        f" | Loss: {loss.item():.4f}"
    )

print("\nSMOKE TEST HOÀN THÀNH")
print("Loss history:", loss_history)

In [ ]:
from pathlib import Path
import torch
import json

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
PILOT_DIR = PROJECT_DIR / "datasets" / "efficientnet_pilot_37"

CHECKPOINT_DIR = PROJECT_DIR / "models" / "pilot_37"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = (
    CHECKPOINT_DIR / "efficientnet_b0_smoke_test.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_names": class_names,
        "num_classes": len(class_names),
        "loss_history": loss_history,
        "training_type": "smoke_test",
        "pretrained": False,
        "input_size": 224
    },
    checkpoint_path
)

print("CHECKPOINT SAVED")
print("Path:", checkpoint_path)
print(
    "Size (MB):",
    round(checkpoint_path.stat().st_size / 1024**2, 2)
)

In [ ]:
from pathlib import Path
import torch
from torchvision.models import EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

cache_dir = Path(torch.hub.get_dir()) / "checkpoints"
weight_filename = weights.url.rsplit("/", 1)[-1]
weight_path = cache_dir / weight_filename

print("===== PRETRAINED WEIGHTS =====")
print("Filename:", weight_filename)
print("Cache directory:", cache_dir)
print("Available locally:", weight_path.is_file())

if weight_path.is_file():
    print(
        "File size (MB):",
        round(weight_path.stat().st_size / 1024**2, 2)
    )
else:
    print("Chưa có pretrained weights trong cache PyTorch.")

In [ ]:
from pathlib import Path
import json
import torch
import torch.nn as nn

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
PILOT_DIR = PROJECT_DIR / "datasets" / "efficientnet_pilot_37"

CHECKPOINT_DIR = PROJECT_DIR / "models" / "pilot_37"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

with open(
    PILOT_DIR / "class_names.json",
    "r",
    encoding="utf-8"
) as f:
    class_names = json.load(f)

device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))

# Nạp backbone pretrained ImageNet
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Thay classifier ImageNet bằng classifier 37 lớp
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(
    in_features,
    len(class_names)
)

# Đóng băng backbone để giảm chi phí CPU
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("===== PRETRAINED MODEL =====")
print("Device:", device)
print("Pretrained:", weights.name)
print("Classes:", len(class_names))
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Backbone frozen:", True)

In [ ]:
import pandas as pd
from PIL import Image, ImageOps
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

train_df = pd.read_csv(
    PILOT_DIR / "train_manifest.csv",
    low_memory=False
)

class_to_idx = {
    name: idx
    for idx, name in enumerate(class_names)
}

# Chọn tối đa 20 ảnh thật cho mỗi lớp
sample_df = pd.concat(
    [
        part.sample(
            n=min(20, len(part)),
            random_state=42
        )
        for _, part in train_df.groupby("target_class")
    ],
    ignore_index=True
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class PilotDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        with Image.open(row["path"]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
            image = transform(image)

        label = class_to_idx[row["target_class"]]
        return image, label

sample_dataset = PilotDataset(sample_df)

sample_loader = DataLoader(
    sample_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

print("===== SAMPLE DATASET =====")
print("Tổng ảnh:", len(sample_dataset))
print("Số lớp:", sample_df["target_class"].nunique())
print("Batch size:", 8)
print(
    "Lớp ít ảnh nhất:",
    sample_df["target_class"].value_counts().min()
)

In [ ]:
import time
import torch.optim as optim
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

MAX_STEPS = 10
loss_history_pretrained = []

# Giữ backbone ở eval để BatchNorm không cập nhật
model.features.eval()
model.classifier.train()

print("===== PRETRAINED TRAINING TEST =====")
print("Max steps:", MAX_STEPS)

start_time = time.time()

progress = tqdm(
    sample_loader,
    total=MAX_STEPS,
    desc="Pretrained CPU training"
)

for step, (images, labels) in enumerate(progress):

    if step >= MAX_STEPS:
        break

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    # Backbone frozen: chỉ tính features, không tính gradient
    with torch.no_grad():
        features = model.features(images)
        features = model.avgpool(features)
        features = torch.flatten(features, 1)

    outputs = model.classifier(features)

    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    current_loss = float(loss.item())
    loss_history_pretrained.append(current_loss)

    progress.set_postfix(
        loss=f"{current_loss:.4f}"
    )

elapsed = time.time() - start_time

print("\n===== KẾT QUẢ =====")
print("Số batch:", len(loss_history_pretrained))
print("Loss đầu:", round(loss_history_pretrained[0], 4))
print("Loss cuối:", round(loss_history_pretrained[-1], 4))
print("Thời gian:", round(elapsed, 1), "giây")
print(
    "Trung bình/batch:",
    round(elapsed / len(loss_history_pretrained), 2),
    "giây"
)

In [ ]:
checkpoint_path = (
    CHECKPOINT_DIR / "efficientnet_b0_pretrained_pilot_37.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_names": class_names,
        "num_classes": len(class_names),
        "loss_history": loss_history_pretrained,
        "steps_completed": len(loss_history_pretrained),
        "pretrained": True,
        "backbone_frozen": True,
        "training_type": "pilot_10_batches",
        "input_size": 224,
        "normalization": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225]
        }
    },
    checkpoint_path
)

print("CHECKPOINT SAVED")
print("Path:", checkpoint_path)
print(
    "Size MB:",
    round(checkpoint_path.stat().st_size / 1024**2, 2)
)

In [ ]:
from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models import efficientnet_b0

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
PILOT_DIR = PROJECT_DIR / "datasets" / "efficientnet_pilot_37"
CHECKPOINT_DIR = PROJECT_DIR / "models" / "pilot_37"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CHECKPOINT = (
    CHECKPOINT_DIR / "efficientnet_b0_pretrained_pilot_37.pth"
)

EPOCH_CHECKPOINT = (
    CHECKPOINT_DIR / "efficientnet_b0_epoch_training.pth"
)

with open(
    PILOT_DIR / "class_names.json",
    encoding="utf-8"
) as f:
    class_names = json.load(f)

device = torch.device("cpu")
torch.set_num_threads(4)

model = efficientnet_b0(weights=None)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(class_names)
)

for p in model.features.parameters():
    p.requires_grad = False

model = model.to(device)

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

criterion = nn.CrossEntropyLoss()

# Ưu tiên checkpoint epoch gần nhất.
# Nếu chưa có, tiếp tục từ checkpoint 10 batch.
load_path = (
    EPOCH_CHECKPOINT
    if EPOCH_CHECKPOINT.exists()
    else BASE_CHECKPOINT
)

checkpoint = torch.load(
    load_path,
    map_location="cpu",
    weights_only=True
)

if checkpoint["class_names"] != class_names:
    raise ValueError("Thứ tự nhãn trong checkpoint không khớp!")

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

completed_epochs = int(checkpoint.get("epoch", 0))
history = list(checkpoint.get("epoch_history", []))

print("===== TRAINING CONFIG =====")
print("Device:", device)
print("Classes:", len(class_names))
print("Checkpoint:", load_path.name)
print("Epoch đã hoàn thành:", completed_epochs)
print("Backbone frozen:", True)

In [ ]:
from PIL import Image, ImageOps
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd

train_df = pd.read_csv(
    PILOT_DIR / "train_manifest.csv",
    low_memory=False
)

class_to_idx = {
    name: i for i, name in enumerate(class_names)
}

# Tái tạo chính xác danh sách mẫu theo seed trước đó
sample_df = pd.concat(
    [
        part.sample(
            n=min(20, len(part)),
            random_state=42
        )
        for _, part in train_df.groupby("target_class")
    ],
    ignore_index=True
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class FoodPilotDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        with Image.open(row["path"]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
            image = transform(image)

        label = class_to_idx[row["target_class"]]

        return image, label

sample_dataset = FoodPilotDataset(sample_df)

sample_loader = DataLoader(
    sample_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

print("Ảnh huấn luyện mẫu:", len(sample_dataset))
print("Số batch mỗi epoch:", len(sample_loader))
print("Số lớp:", sample_df["target_class"].nunique())

In [ ]:
import os
import time
import torch
from tqdm.auto import tqdm

EXTRA_EPOCHS = 2
target_epoch = completed_epochs + EXTRA_EPOCHS

print("===== BẮT ĐẦU TRAINING =====")
print("Từ epoch:", completed_epochs + 1)
print("Đến epoch:", target_epoch)

for epoch in range(completed_epochs + 1, target_epoch + 1):

    start = time.time()

    # Không cập nhật BatchNorm của pretrained backbone
    model.features.eval()
    model.classifier.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    progress = tqdm(
        sample_loader,
        desc=f"Epoch {epoch}/{target_epoch}"
    )

    for images, labels in progress:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            features = model.features(images)
            features = model.avgpool(features)
            features = torch.flatten(features, 1)

        outputs = model.classifier(features)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{total_loss / total_samples:.4f}",
            acc=f"{100 * total_correct / total_samples:.2f}%"
        )

    epoch_loss = total_loss / total_samples
    epoch_acc = 100 * total_correct / total_samples
    elapsed = time.time() - start

    history.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "train_accuracy": epoch_acc,
        "seconds": elapsed
    })

    # Lưu vào file tạm rồi thay checkpoint chính
    temp_path = EPOCH_CHECKPOINT.with_suffix(".tmp")

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "class_names": class_names,
            "epoch_history": history,
            "training_type": "sample_cpu_epoch_test",
            "pretrained": True,
            "backbone_frozen": True,
            "input_size": 224
        },
        temp_path
    )

    os.replace(temp_path, EPOCH_CHECKPOINT)

    print(
        f"\nEpoch {epoch} hoàn thành"
        f" | Loss: {epoch_loss:.4f}"
        f" | Train Accuracy: {epoch_acc:.2f}%"
        f" | Thời gian: {elapsed:.1f} giây"
    )

print("\nĐÃ LƯU CHECKPOINT:", EPOCH_CHECKPOINT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

display(history_df)

plt.figure(figsize=(7, 4))
plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("EfficientNet-B0 Pilot - Training Loss")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy (%)")
plt.title("EfficientNet-B0 Pilot - Training Accuracy")
plt.grid(alpha=0.3)
plt.show()

history_df.to_csv(
    CHECKPOINT_DIR / "epoch_training_history.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
import json

ROOT = Path(r"C:\Users\ASUS\SIC-Food")

PILOT_DIR = ROOT / "datasets" / "efficientnet_pilot_37"
V5_DIR = ROOT / "datasets" / "raw" / "state_sources" / "extension_v5"

ARCHIVE_DIR = V5_DIR / "archives"
MANUAL_DIR = V5_DIR / "manual_images"
REPORT_DIR_V5 = V5_DIR / "reports"

priority_classes = [
    "broccoli__visible_defect",
    "shrimp__normal_appearance",
    "shrimp__visible_defect",
    "eggplant__visible_defect",
    "eggplant__normal_appearance",
    "broccoli__normal_appearance",
    "onion__visible_defect"
]

for folder in [ARCHIVE_DIR, MANUAL_DIR, REPORT_DIR_V5]:
    folder.mkdir(parents=True, exist_ok=True)

for class_name in priority_classes:
    (MANUAL_DIR / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

with open(
    PILOT_DIR / "class_names.json",
    encoding="utf-8"
) as f:
    existing_classes = json.load(f)

missing_classes = [
    "broccoli__visible_defect",
    "shrimp__normal_appearance",
    "shrimp__visible_defect"
]

final_classes = sorted(
    set(existing_classes) | set(missing_classes)
)

assert len(final_classes) == 40, (
    f"Phát hiện {len(final_classes)} lớp, không phải 40"
)

with open(
    REPORT_DIR_V5 / "class_names_40_planned.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_classes,
        f,
        ensure_ascii=False,
        indent=2
    )

print("===== EFFICIENTNET V5 =====")
print("Lớp hiện có:", len(existing_classes))
print("Lớp mục tiêu:", len(final_classes))
print("Lớp cần bổ sung:", missing_classes)
print("Thư mục ZIP:", ARCHIVE_DIR)
print("Thư mục ảnh mới:", MANUAL_DIR)
print("READY!")

In [ ]:
from pathlib import Path
import zipfile
from collections import Counter
import yaml

archives = sorted(ARCHIVE_DIR.glob("*.zip"))

print("===== KIỂM TRA ARCHIVES V5 =====")
print("Số file ZIP:", len(archives))

for archive_path in archives:

    print("\n" + "=" * 60)
    print("ARCHIVE:", archive_path.name)

    with zipfile.ZipFile(archive_path, "r") as z:
        names = [
            n for n in z.namelist()
            if not n.endswith("/")
        ]

        image_files = [
            n for n in names
            if Path(n).suffix.lower()
            in {".jpg", ".jpeg", ".png", ".webp"}
        ]

        label_files = [
            n for n in names
            if n.lower().endswith(".txt")
            and "/labels/" in n.replace("\\", "/").lower()
        ]

        yaml_files = [
            n for n in names
            if Path(n).name.lower()
            in {"data.yaml", "data.yml"}
        ]

        print("Ảnh:", len(image_files))
        print("Label TXT:", len(label_files))
        print("YAML:", yaml_files)

        for yaml_name in yaml_files[:3]:
            try:
                config = yaml.safe_load(
                    z.read(yaml_name).decode("utf-8-sig")
                )

                print("\nCấu hình:", yaml_name)
                print("Classes:", config.get("names"))
                print("Number of classes:", config.get("nc"))

            except Exception as e:
                print("Lỗi YAML:", str(e))

        split_counts = Counter()

        for image_name in image_files:
            parts = image_name.replace("\\", "/").lower().split("/")

            for split in ["train", "valid", "val", "test"]:
                if split in parts:
                    split_counts[split] += 1
                    break
            else:
                split_counts["unknown"] += 1

        print("\nẢnh theo split:")
        print(dict(split_counts))